# Musanga V2 — Notebook 04 | Illumination-robust U-Net pilot

**Pilot then full four-site LOSO within this SAME notebook:** Train the original ResNet-18 U-Net on three forests and evaluate on the fourth, comparing:

- **M0:** native RGB + historical geometric/mild brightness augmentation
- **M1:** native RGB + realistic stronger illumination/gamma/contrast/smooth-shadow augmentation
- **M2:** M1 + per-site RGB gain referenced to **three training forests**

Sections 00–08 run the small pilot; Sections 09–11 run full four-site LOSO with all eligible crowns, M0/M1 and resumable fold checkpoints.

Default pilot LOSO holdout: **Baego**. This is a fast **single-fold pilot**, not full four-fold LOSO. Individual crown-centred native-resolution patches are used for cheap iteration, and sample sizes are capped. Only annotated crown pixels contribute to loss. Heldout test crowns never select epochs or hyperparameters. Checkpoints are resumable and per-batch/per-epoch progress is printed.

**Important M2 caveat:** Training reference median is fit on the three training forests, but M2 also reads the heldout site's **unlabelled** RGB median to compute its gain; therefore it represents *unsupervised test-image adaptation*, not strict fixed preprocessing. The distinction must be retained in later LOSO reporting.

**Luki:** completely excluded from Notebook 04 fitting and test selection. Its current merged 364-crown reference and forthcoming dark-area additions belong to versioned external evaluation later. No need to touch earlier source imagery or rerun earlier notebook stages to train this pilot.

**Artifacts:** exclusively `MyDrive/Colab Notebooks/Musanga/KVO_work/V2/04_illumination_robust_training/`, with a fingerprinted pilot run directory. No full corrected mosaics or legacy checkpoints are overwritten.

**Execution:** Select a Colab GPU before Section 06. The first run generates and compresses a patch cache once, reusable across all three variants. For an entirely fresh GPU runtime, **run Sections 12–14 only** to train the final four-site M1 model and evaluate Luki. Sections 00–11 are the historical pilot/LOSO experiments and do not need to be rerun. The final model is trained independently from LOSO and is checked against the expanded frozen Luki reference.

## 00 — Setup, pilot budget and source locks

Configure a small **single held-out-forest pilot** first. Default held-out Baego; the other three forests provide spatial training/validation. All outputs are versioned under `KVO_work/V2/04_illumination_robust_training/`. Luki is not loaded or used for fitting.

In [ ]:
%pip -q install rasterio geopandas pyogrio segmentation-models-pytorch scikit-learn tqdm matplotlib
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,rasterio
from rasterio.windows import Window
from rasterio.features import rasterize,geometry_mask
from shapely.geometry import box
from sklearn.metrics import roc_auc_score,average_precision_score,precision_score,recall_score,f1_score,balanced_accuracy_score
from tqdm.auto import tqdm
from IPython.display import display
import segmentation_models_pytorch as smp
import torch,torch.nn as nn
from torch.utils.data import Dataset,DataLoader
import json,hashlib,random,time,math,gc,sys
from google.colab import drive
MY=Path("/content/drive/MyDrive")
if not MY.exists():drive.mount("/content/drive")
ROOT=MY/"Colab Notebooks"/"Musanga"
GEO=ROOT/"data"/"Geo_Ref";AD=GEO/"Annotations"
V2=ROOT/"KVO_work"/"V2"
REF=V2/"01_reference_splits"
HARM=V2/"03_illumination_harmonization"
OUT=V2/"04_illumination_robust_training"
OUT.mkdir(parents=True,exist_ok=True)
CACHE=OUT/"pilot_patch_cache";CACHE.mkdir(exist_ok=True)
SITES={
 "MOS_1_2":("MOS_1_2_kvo.tif","Tree_Annotations_MOS_1_2_kvo.shp"),
 "Yoko":("Yoko_kvo_georef.tif","Tree_Annotations_Yoko_kvo_georef.shp"),
 "Yangambi":("Yangambi_georef.tif","Tree_Annotation_Yangambi_georef.shp"),
 "Baego":("Baego_georef.tif","Tree_Annotation_Baego.shp")}
# PILOT defaults: change only after inspecting first comparisons.
HELDOUT="Baego"
PILOT_TRAIN_CROWNS_PER_SITE=240  # cap across classes, balanced sampling
PILOT_VAL_CROWNS_PER_SITE=90
PILOT_TEST_CROWNS=180
PATCH=256
MAX_EPOCHS=4
PATIENCE=3
BATCH_SIZE=12
SEED=20261013
LR=2e-4
NUM_WORKERS=0  # Colab-safe; on-demand augmentation cheap, avoids multiprocessing state problems
IGNORE=255
MIN_AREA_M2=5.0
RUN_VARIANTS=["M0_native_historical","M1_native_photometric","M2_sitegain_photometric"]
FORCE_REBUILD_CACHE=False
FORCE_RESTART=False
RUN_TRAINING=True
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
assert HELDOUT in SITES
assert (REF/"01_spatial_split_summary.csv").exists(), "Run V2 Notebook 01 first"
assert (HARM/"03_site_luminance_and_fixed_gains.csv").exists(), "Run V2 Notebook 03 Section 02"
if DEVICE=="cpu":
    print("WARNING: CPU detected — select a GPU runtime before training.")
else: print("GPU:",torch.cuda.get_device_name(0))
print("PILOT heldout:",HELDOUT,"epochs:",MAX_EPOCHS,"output:",OUT)
def sha_file(path):
    stat=path.stat()
    return dict(path=str(path),size_bytes=stat.st_size,mtime_ns=stat.st_mtime_ns)
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
seed_all(SEED)


## 01 — Load immutable V2 crown splits

Reconstruct exactly the crowns from V2 Notebook 01 and join the saved spatial split IDs. Invalid geometries stay excluded. This stage does not merge, edit or read any Luki labels; forthcoming dark-area annotations will be handled by a separately versioned external-evaluation update.

In [ ]:
ANN={};SOURCE_FINGERPRINT={}
for site,(filename,shp) in SITES.items():
    rp=GEO/filename;ap=AD/shp
    assert rp.exists() and ap.exists(),site
    with rasterio.open(rp) as src:
        crs=src.crs;bounds=box(*src.bounds)
        assert crs and crs.is_projected
    g=gpd.read_file(ap)
    assert g.crs is not None and "Class_ID" in g.columns
    if g.crs!=crs:g=g.to_crs(crs)
    valid=g.geometry.notna()&(~g.geometry.is_empty)&g.geometry.is_valid
    g=g.loc[valid].copy()
    numerical=pd.to_numeric(g.Class_ID,errors="coerce")
    g=g.loc[numerical.notna()].copy()
    g["class_id"]=numerical.loc[g.index].astype(int)
    g["label"]=(g.class_id==1).astype(np.uint8)
    g["area_m2"]=g.geometry.area
    g=g.loc[(g.area_m2>=MIN_AREA_M2)&g.geometry.intersects(bounds)].copy()
    g["crown_id"]=site+":"+g.index.astype(str)
    reg=pd.read_csv(REF/f"01_{site}_split_register.csv",dtype={"crown_id":"string"})
    assert reg.crown_id.is_unique
    assert set(g.crown_id)==set(reg.crown_id),f"{site} annotation differs from locked Notebook 01 split"
    g=g.merge(reg[["crown_id","split","spatial_block"]],on="crown_id",how="left",validate="one_to_one")
    assert g.split.isin(["training","validation","buffered"]).all()
    ANN[site]=g
    SOURCE_FINGERPRINT[site]=dict(image=sha_file(rp),annotation=sha_file(ap),
                                   split=sha_file(REF/f"01_{site}_split_register.csv"))
    print(site,":",len(g),"crowns; train",sum(g.split=="training"),
          "val",sum(g.split=="validation"),"buffered",sum(g.split=="buffered"))
print("LOSO TEST:",HELDOUT,"uses all annotation classes but only a bounded pilot subsample.")


## 02 — Honest training-only global normalization reference

For M2, calculate the shared reference using only the **three training forests**. Each site's own image brightness index is estimated without species labels. Applying an image-specific gain to the unseen held-out forest is **unsupervised target-image adaptation**, and must be reported as such rather than called strict fixed-preprocessing transfer. M0 and M1 are pure native-RGB transfer.

In [ ]:
stats=pd.read_csv(HARM/"03_site_luminance_and_fixed_gains.csv")
assert set(SITES).issubset(set(stats.site))
training_sites=[s for s in SITES if s!=HELDOUT]
LUMINANCE_MEDIAN={row.site:float(row.site_luminance_median) for _,row in stats.iterrows()}
reference=float(np.median([LUMINANCE_MEDIAN[s] for s in training_sites]))
GLOBAL_GAIN_BOUNDS=(.85,1.15)
GAIN={s:float(np.clip(reference/max(LUMINANCE_MEDIAN[s],1e-6),*GLOBAL_GAIN_BOUNDS)) for s in SITES}
print("Per-fold reference from training sites ONLY:",training_sites,round(reference,4))
print("Per-site gains (test site's own unlabelled RGB statistics imply unsupervised adaptation):",GAIN)
assert HELDOUT not in training_sites


## 03 — Deterministic pilot samples and native-resolution patch cache

Build one shared patch cache per fold from identical source crowns for **all model variants**, with 256 px imagery at each forest's native GSD. Labels are rasterized only from the same training/validation split (all other pixels ignored). Sampling is class-balanced. No full-mosaic masks are constructed, and caches are fingerprinted and reusable.

In [ ]:
def choose_crowns(g,split,n,seed):
    selected=g[g.split==split].copy()
    if n is None or len(selected)<=n:return selected
    rng=np.random.default_rng(seed)
    # Sample both classes; avoid extreme imbalance and keep full indices.
    half=n//2
    p=selected[selected.label==1].index.to_numpy()
    q=selected[selected.label==0].index.to_numpy()
    rng.shuffle(p);rng.shuffle(q)
    p=p[:min(len(p),half)]
    q=q[:min(len(q),n-len(p))]
    if len(p)+len(q)<n:
        remaining=selected.index.difference(np.concatenate([p,q])).to_numpy()
        rng.shuffle(remaining)
        ids=np.concatenate([p,q,remaining[:n-len(p)-len(q)]])
    else:ids=np.concatenate([p,q])
    return selected.loc[ids].copy()

pilot_indices={}
for j,site in enumerate(training_sites):
    g=ANN[site]
    pilot_indices[(site,"training")]=choose_crowns(g,"training",PILOT_TRAIN_CROWNS_PER_SITE,SEED+10*j)
    pilot_indices[(site,"validation")]=choose_crowns(g,"validation",PILOT_VAL_CROWNS_PER_SITE,SEED+100+10*j)
TEST_CROWNS=choose_crowns(ANN[HELDOUT].assign(split="external"),"external",
                          PILOT_TEST_CROWNS,SEED+555)
assert TEST_CROWNS.label.nunique()==2
print("Pilot train",sum(len(pilot_indices[s,"training"]) for s in training_sites),
      "val",sum(len(pilot_indices[s,"validation"]) for s in training_sites),
      "heldout test",len(TEST_CROWNS))
configuration=dict(heldout=HELDOUT,seed=SEED,patch=PATCH,train_cap=PILOT_TRAIN_CROWNS_PER_SITE,
    val_cap=PILOT_VAL_CROWNS_PER_SITE,test_cap=PILOT_TEST_CROWNS,
    source=SOURCE_FINGERPRINT,ref=sha_file(HARM/"03_site_luminance_and_fixed_gains.csv"),
    experiment="v2_pilot_centered_crown_patch_v1")
RUN_ID=hashlib.sha256(json.dumps(configuration,sort_keys=True).encode()).hexdigest()[:12]
RUN_OUT=OUT/f"{HELDOUT}_{RUN_ID}";RUN_OUT.mkdir(exist_ok=True)
NPZ=CACHE/f"pilot_{HELDOUT}_{RUN_ID}.npz"
def centered_window(g,patch,src):
    cx,cy=g.centroid.x,g.centroid.y
    row,col=rasterio.transform.rowcol(src.transform,cx,cy)
    return Window(int(col)-patch//2,int(row)-patch//2,patch,patch)

def generate_cache03():
    xx=[];yy=[];ss=[];sites=[];ids=[]
    # A crop is selected using a crown centroid, but loss includes only the anchor
    # crown's own pixels. This avoids assigning a neighboring split's labels.
    for site in training_sites:
        with rasterio.open(GEO/SITES[site][0]) as src:
            for split_code,split in ((0,"training"),(1,"validation")):
                g=pilot_indices[site,split]
                for _,rec in tqdm(g.iterrows(),total=len(g),desc=f"{site}/{split}",leave=False):
                    w=centered_window(rec.geometry,PATCH,src)
                    rgb=src.read([1,2,3],window=w,boundless=True,fill_value=0)
                    arr=np.moveaxis(rgb,0,-1).astype(np.uint8)
                    original=arr
                    valid=~((original==0).all(axis=2)|
                            (original==255).all(axis=2)|
                            ((original[:,:,0]==0)&(original[:,:,1]==0)&(original[:,:,2]==255)))
                    mask=rasterize([(rec.geometry,int(rec.label))],out_shape=(PATCH,PATCH),
                                   transform=src.window_transform(w),fill=IGNORE,dtype="uint8")
                    mask[~valid]=IGNORE
                    if int((mask!=IGNORE).sum())<12:continue
                    xx.append(arr);yy.append(mask);ss.append(split_code)
                    sites.append(site);ids.append(str(rec.crown_id))
    assert xx and len(set(ss))==2
    X=np.stack(xx);Y=np.stack(yy);S=np.array(ss,np.uint8)
    SI=np.array(sites);CI=np.array(ids)
    print("Caching",len(X),"patches:",round(X.nbytes/1e6,1),"MB raw RGB",flush=True)
    np.savez_compressed(NPZ,X=X,Y=Y,S=S,SI=SI,CI=CI)
    return X,Y,S,SI,CI
if NPZ.exists() and not FORCE_REBUILD_CACHE:
    obj=np.load(NPZ,allow_pickle=False)
    X,Y,S,SI,CI=[obj[k] for k in ["X","Y","S","SI","CI"]]
    print("Reused fingerprinted pilot cache",NPZ)
else:
    X,Y,S,SI,CI=generate_cache03()
assert len(X)==len(Y)==len(S)==len(SI)==len(CI)
assert (S==0).any() and (S==1).any()
print("Patch cache:",len(X),"train:",int((S==0).sum()),"val:",int((S==1).sum()))
pd.DataFrame(dict(site=SI,split=np.where(S==0,"training","validation"),
                  crown_id=CI)).to_csv(RUN_OUT/"04_pilot_patch_registry.csv",index=False)


## 04 — Model, preprocessing and photometric augmentation

Keep the original ResNet-18 U-Net and ImageNet RGB normalization. Compare **M0** historical modest brightness jitter, **M1** randomized brightness/gamma/contrast/channel shifts and smooth shadows, **M2** exactly M1 plus a constant training-site gain. All share architecture, seed, sampled training crowns and optimization settings. Validation never uses random augmentation.

In [ ]:
MEAN=np.array([.485,.456,.406],dtype=np.float32)[None,None,:]
STD=np.array([.229,.224,.225],dtype=np.float32)[None,None,:]
def shadow04(x,rng):
    # Spatially broad and smooth illumination change on the crop.
    # Default local mask variation is differentiable and multiplicative, not recoloring.
    h,w=x.shape[:2]
    yy,xx=np.mgrid[:h,:w].astype(np.float32)
    theta=rng.uniform(0,np.pi)
    normal=(np.cos(theta)*(xx-w/2)+np.sin(theta)*(yy-h/2))
    offset=rng.uniform(-.35,.35)*max(h,w)
    edge_width=rng.uniform(20,90)
    transition=np.clip((normal-offset)/edge_width,-12,12)
    weights=1/(1+np.exp(-transition))
    darkness=rng.uniform(.45,.90)
    return np.clip(x*(darkness+(1-darkness)*weights[:,:,None]),0,1)
class PilotDataset04(Dataset):
    def __init__(self,which,variant,epoch=0):
        self.indices=np.flatnonzero(S==which)
        self.variant=variant;self.epoch=epoch;self.training=(which==0)
    def __len__(self):return len(self.indices)
    def __getitem__(self,j):
        index=int(self.indices[j])
        img=X[index].astype(np.float32)/255.
        mask=Y[index].copy()
        site=str(SI[index])
        if self.variant=="M2_sitegain_photometric":
            img=np.clip(img*GAIN[site],0,1)
        if self.training:
            rng=np.random.default_rng(SEED+self.epoch*1_000_003+index*101)
            if rng.random()<.5:img=img[:,::-1,:].copy();mask=mask[:,::-1].copy()
            if rng.random()<.5:img=img[::-1,:,:].copy();mask=mask[::-1,:].copy()
            k=int(rng.integers(0,4))
            img=np.rot90(img,k).copy();mask=np.rot90(mask,k).copy()
            if self.variant=="M0_native_historical":
                img=np.clip(img*rng.uniform(.90,1.10),0,1)
            else:
                img=np.clip(img*rng.uniform(.70,1.30),0,1)
                img=np.clip((img-.5)*rng.uniform(.80,1.20)+.5,0,1)
                img=np.power(img,float(rng.uniform(.75,1.35)))
                img=np.clip(img*rng.uniform(.92,1.08,size=(1,1,3)),0,1)
                if rng.random()<.55:img=shadow04(img,rng)
        valid=mask!=IGNORE
        if valid.sum()==0:raise RuntimeError("Empty training patch after source QA")
        tensor=np.moveaxis(((img-MEAN)/STD).astype(np.float32),-1,0).copy()
        return torch.from_numpy(tensor),torch.from_numpy(mask.astype(np.int64))
def new_model04():
    return smp.Unet(encoder_name="resnet18",encoder_weights="imagenet",
                    in_channels=3,classes=1).to(DEVICE)
def make_loader04(code,variant,epoch):
    ds=PilotDataset04(code,variant,epoch)
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=(code==0),
        pin_memory=DEVICE=="cuda",num_workers=NUM_WORKERS)
print("Each variant has identical architecture, patch inventory and RNG seed.")
print("M2 fold reference learned only from:",training_sites)


## 05 — Fast smoke test (before GPU training)

Validate that each treatment creates finite tensors and that masked targets contain both species classes. Detect configuration errors **before** allocating long GPU runs.

In [ ]:
assert len(set(SI))==len(training_sites)
for variant in RUN_VARIANTS:
    tr=PilotDataset04(0,variant,epoch=1)
    va=PilotDataset04(1,variant,epoch=0)
    a,b=tr[0];c,d=va[0]
    assert a.shape==(3,PATCH,PATCH) and c.shape==(3,PATCH,PATCH)
    assert torch.isfinite(a).all() and torch.isfinite(c).all()
    assert set(torch.unique(b).tolist()).issubset({0,1,IGNORE})
    print(variant,"train batches",math.ceil(len(tr)/BATCH_SIZE),
          "val batches",math.ceil(len(va)/BATCH_SIZE),
          "x range",round(float(a.min()),2),round(float(a.max()),2))
print("Smoke test passed.")


## 06 — Resumable pilot training with per-epoch progress

Run M0/M1/M2 sequentially with an identical pilot budget. Checkpoints are saved **every epoch**, including optimizer, scheduler-free training state, AMP scaler, early-stopping count and best validation AP; reruns resume instead of starting over. Console output is flushed each epoch. No held-out-forest labels are used for checkpoint selection.

In [ ]:
def score_valid04(model,loader):
    model.eval();true=[];pred=[]
    with torch.inference_mode():
        for x,y in loader:
            x=x.to(DEVICE,non_blocking=True)
            with torch.amp.autocast("cuda",enabled=DEVICE=="cuda"):
                out=torch.sigmoid(model(x).squeeze(1))
            mask=(y!=IGNORE)
            true.append(y[mask].numpy().astype(np.uint8))
            pred.append(out.detach().cpu()[mask].numpy().astype(np.float32))
    target=np.concatenate(true);prob=np.concatenate(pred)
    return dict(ap=float(average_precision_score(target,prob)) if target.sum() else np.nan,
        auc=float(roc_auc_score(target,prob)) if len(np.unique(target))==2 else np.nan,
        pixels=int(len(target)))
def train_variant04(variant):
    folder=RUN_OUT/variant;folder.mkdir(parents=True,exist_ok=True)
    progress=folder/"last_training_state.pt";bestfile=folder/"best_model.pt"
    history_file=folder/"epoch_history.csv";marker=folder/"pilot_complete.json"
    if marker.exists() and bestfile.exists() and not FORCE_RESTART:
        print("REUSING COMPLETED PILOT:",variant,flush=True)
        return json.loads(marker.read_text())
    seed_all(SEED)
    model=new_model04()
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
    scaler=torch.amp.GradScaler("cuda",enabled=DEVICE=="cuda")
    history=[];start=1;best=-1.;wait=0
    if progress.exists() and not FORCE_RESTART:
        state=torch.load(progress,map_location=DEVICE,weights_only=False)
        assert state["variant"]==variant and state["run_id"]==RUN_ID
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        best=float(state["best"]);wait=int(state["wait"])
        start=int(state["epoch"])+1
        history=state["history"]
        print("RESUMING:",variant,"epoch",start,"best AP",round(best,4),flush=True)
    for epoch in range(start,MAX_EPOCHS+1):
        begin=time.monotonic();model.train();losses=[]
        loader=make_loader04(0,variant,epoch)
        for bi,(x,y) in enumerate(loader,1):
            x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda",enabled=DEVICE=="cuda"):
                logits=model(x).squeeze(1)
                valid=(y!=IGNORE)
                loss=nn.functional.binary_cross_entropy_with_logits(logits[valid],y[valid].float())
            scaler.scale(loss).backward()
            scaler.step(optimizer);scaler.update()
            losses.append(float(loss.item()))
            if bi==1 or bi%25==0 or bi==len(loader):
                print(f"  {variant} epoch {epoch} batch {bi}/{len(loader)} loss {np.mean(losses):.4f}",
                      flush=True)
        validation=score_valid04(model,make_loader04(1,variant,0))
        improved=np.isfinite(validation["ap"]) and validation["ap"]>best
        if improved:
            best=validation["ap"];wait=0
            torch.save(dict(model=model.state_dict(),epoch=epoch,variant=variant,
                            run_id=RUN_ID,best_validation_ap=best),bestfile)
        else:wait+=1
        info=dict(epoch=epoch,train_loss=float(np.mean(losses)),validation_ap=validation["ap"],
                  validation_auc=validation["auc"],validation_pixels=validation["pixels"],
                  seconds=float(time.monotonic()-begin),best_validation_ap=best,wait=wait)
        history.append(info)
        pd.DataFrame(history).to_csv(history_file,index=False)
        torch.save(dict(variant=variant,run_id=RUN_ID,epoch=epoch,
            model=model.state_dict(),optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),
            best=best,wait=wait,history=history),progress)
        print(f"PROGRESS {variant} epoch {epoch}/{MAX_EPOCHS}: loss={info['train_loss']:.4f}, "
              f"AP={info['validation_ap']:.4f}, AUC={info['validation_auc']:.4f}, "
              f"best_AP={best:.4f}, {info['seconds']:.1f}s",flush=True)
        if wait>=PATIENCE:
            print("EARLY STOP:",variant,"after",epoch,"epochs",flush=True);break
    if not bestfile.exists():raise RuntimeError(f"Best checkpoint never saved for {variant}")
    summary=dict(variant=variant,run_id=RUN_ID,heldout=HELDOUT,
                 epochs_completed=len(history),best_validation_ap=best,
                 selection_basis="spatial validation pixels on three training forests",
                 heldout_labels_used_for_selection=False)
    marker.write_text(json.dumps(summary,indent=2))
    del model,optimizer;gc.collect()
    if DEVICE=="cuda":torch.cuda.empty_cache()
    return summary
TRAIN_RESULTS04=[]
if RUN_TRAINING:
    assert DEVICE=="cuda","Change runtime to GPU; refusing long CPU training."
    for variant in RUN_VARIANTS:
        print("\nSTART VARIANT:",variant,flush=True)
        TRAIN_RESULTS04.append(train_variant04(variant))
    display(pd.DataFrame(TRAIN_RESULTS04))
else:print("RUN_TRAINING=False. Smoke test only.")


## 07 — Small independent Baego test; no selection on test

Evaluate each trained variant on an identical, class-balanced sample from the **held-out fourth forest**. Each crown is scored using mean predicted probability within its original labelled polygon on a native-resolution patch. Results are diagnostic (pilot sample), **not full wall-to-wall canopy accuracy or complete LOSO**. Do not choose hyperparameters based on Luki.

In [ ]:
def test_crowns04(model,variant,test_g):
    model.eval();rows=[]
    with rasterio.open(GEO/SITES[HELDOUT][0]) as src:
        for _,rec in tqdm(test_g.iterrows(),total=len(test_g),desc=f"{variant} heldout crowns"):
            w=centered_window(rec.geometry,PATCH,src)
            raw=np.moveaxis(src.read([1,2,3],window=w,boundless=True,fill_value=0),0,-1)
            valid=~((raw==0).all(axis=2)|(raw==255).all(axis=2)|
                ((raw[:,:,0]==0)&(raw[:,:,1]==0)&(raw[:,:,2]==255)))
            interior=geometry_mask([rec.geometry],out_shape=(PATCH,PATCH),
                                    transform=src.window_transform(w),invert=True)&valid
            if interior.sum()<12:continue
            img=raw.astype(np.float32)/255.
            if variant=="M2_sitegain_photometric":
                img=np.clip(img*GAIN[HELDOUT],0,1)
            normalized=np.moveaxis((img-MEAN)/STD,-1,0).astype(np.float32).copy()
            with torch.inference_mode(),torch.amp.autocast("cuda",enabled=DEVICE=="cuda"):
                p=torch.sigmoid(model(torch.from_numpy(normalized[None]).to(DEVICE))).squeeze().float().cpu().numpy()
            score=float(np.mean(p[interior]))
            rows.append(dict(site=HELDOUT,crown_id=str(rec.crown_id),label=int(rec.label),
                predicted_probability=score,valid_pixels=int(interior.sum()),
                variant=variant,experiment="balanced_heldout_crown_pilot"))
    return pd.DataFrame(rows)
TEST_PRED04=[]
if RUN_TRAINING:
    for variant in RUN_VARIANTS:
        folder=RUN_OUT/variant
        bestfile=folder/"best_model.pt"
        assert bestfile.exists(),f"Missing {bestfile}"
        seed_all(SEED);model=new_model04()
        ckpt=torch.load(bestfile,map_location=DEVICE,weights_only=False)
        assert ckpt["run_id"]==RUN_ID and ckpt["variant"]==variant
        model.load_state_dict(ckpt["model"])
        df=test_crowns04(model,variant,TEST_CROWNS)
        assert df.label.nunique()==2
        df.to_csv(folder/"heldout_pilot_crown_predictions.csv",index=False)
        TEST_PRED04.append(df)
        print("EVALUATED",variant,"crowns",len(df),flush=True)
        del model;gc.collect()
        if DEVICE=="cuda":torch.cuda.empty_cache()
    HOLDOUT04=pd.concat(TEST_PRED04,ignore_index=True)
    HOLDOUT04.to_csv(RUN_OUT/"04_heldout_pilot_crown_predictions.csv",index=False)


## 08 — Compare pilot results and save handoff

Compare validation-selected checkpoints on a held-out forest, alongside run-time budgets. Keep results labelled as **pilot**, and make no claims about Luki or wall-to-wall canopy cover. Subsequent complete four-fold LOSO remains the authority; revised Luki dark-area annotations must have explicit reference versions.

In [ ]:
def summarize04(d):
    y=d.label.to_numpy(dtype=int)
    p=d.predicted_probability.to_numpy(float)
    q=(p>=.5).astype(int)
    return dict(n=len(y),musanga=int(y.sum()),other=int((1-y).sum()),
        auc=float(roc_auc_score(y,p)),average_precision=float(average_precision_score(y,p)),
        precision=float(precision_score(y,q,zero_division=0)),
        recall=float(recall_score(y,q,zero_division=0)),
        balanced_accuracy=float(balanced_accuracy_score(y,q)),
        f1=float(f1_score(y,q,zero_division=0)),
        false_positives=int(((y==0)&(q==1)).sum()),
        false_negatives=int(((y==1)&(q==0)).sum()))
if RUN_TRAINING:
    SUMMARY04=pd.DataFrame([dict(variant=v,**summarize04(HOLDOUT04[HOLDOUT04.variant==v]))
                           for v in RUN_VARIANTS])
    SUMMARY04.to_csv(RUN_OUT/"04_heldout_pilot_comparison.csv",index=False)
    display(SUMMARY04.round(3))
manifest=dict(stage="Musanga V2 Notebook 04: pilot U-Net",
    heldout=HELDOUT,training_sites=training_sites,
    architecture="segmentation_models_pytorch.Unet resnet18 imagenet",
    spatial_splits_source=str(REF),source_files=SOURCE_FINGERPRINT,
    run_id=RUN_ID,run_dir=str(RUN_OUT),
    patch_size=PATCH,seed=SEED,
    training_caps_per_site=PILOT_TRAIN_CROWNS_PER_SITE,
    validation_caps_per_site=PILOT_VAL_CROWNS_PER_SITE,
    heldout_crown_pilot_cap=PILOT_TEST_CROWNS,
    training_epochs=MAX_EPOCHS,patience=PATIENCE,variants=RUN_VARIANTS,
    normalization_reference_sites=training_sites,
    normalization_reference_luminance=reference,site_gains=GAIN,
    m2_interpretation="unlabeled held-out image statistics used for site gain: unsupervised target adaptation",
    heldout_selection_policy="test labels never choose checkpoint or photometric settings",
    luki_policy="Luki excluded from model fitting, checkpoint selection and pilot testing; new dark annotations are future versioned external references",
    spatial_qa="single-crown labelled patches, all other pixels ignored, split buffers inherited from Notebook 01",
    output_policy="All generated files exclusively under Musanga/KVO_work/V2")
(RUN_OUT/"04_pilot_manifest.json").write_text(json.dumps(manifest,indent=2))
print("Pilot run:",RUN_ID)
print("Outputs:",RUN_OUT)
print("NEXT: inspect pilot results and decide whether full four-fold LOSO is warranted.")


## 09 — Full four-site LOSO: configuration and safe state isolation

This is **the full experiment inside Notebook 04**, not a new notebook. It uses all eligible Notebook 01 **training** and **validation** crowns in each of the four LOSO folds, and tests **all eligible held-out crowns**, with no artificial class balancing. Two variants: M0 original modest augmentation, M1 stronger photometric augmentation. Pixel-level validation AP selects checkpoints; independent held-out **crown-level ROC-AUC and AP** are primary ranking metrics. Additional F1 at 0.5 is descriptive only.

**Reproducibility and efficiency:** reuse Section 03/04 cache and U-Net functions; write into a separate `full_LOSO/` folder; freeze pilot globals after this stage; checkpoints every epoch and short streamed training progress. Default maximum 20 epochs and early stopping after four non-improving epochs. Training-crown **centre crops** approximate sampled crown recognition, not wall-to-wall U-Net pixel evaluation. Luki remains untouched.

In [ ]:
# Run the existing pilot once, then run Sections 09–11 to execute full LOSO.
RUN_FULL_LOSO=True
FULL_FOLDS=["Baego","MOS_1_2","Yoko","Yangambi"]
FULL_VARIANTS=["M0_native_historical","M1_native_photometric"]
FULL_MAX_EPOCHS=20
FULL_PATIENCE=4
FULL_BATCH_SIZE=12
FULL_ROOT=OUT/"full_LOSO"
FULL_ROOT.mkdir(parents=True,exist_ok=True)
assert len(FULL_FOLDS)==4 and set(FULL_FOLDS)==set(SITES)
assert set(FULL_VARIANTS).issubset(set(RUN_VARIANTS))
FULL_SUMMARY=FULL_ROOT/"04_full_LOSO_fold_metrics.csv"
# Sections 03–08 deliberately used shared notebook globals for simple pilot reuse.
# Snapshot these before substituting a complete fold and restore in Section 11.
PILOT_GLOBALS04={k:globals()[k] for k in [
    "HELDOUT","training_sites","reference","GAIN","RUN_ID","RUN_OUT",
    "NPZ","pilot_indices","TEST_CROWNS","X","Y","S","SI","CI",
    "RUN_VARIANTS","MAX_EPOCHS","PATIENCE","BATCH_SIZE","FORCE_RESTART"]}
print("Full LOSO folds:",FULL_FOLDS, "| models:",FULL_VARIANTS,flush=True)
print("Max epochs:",FULL_MAX_EPOCHS,"early-stop patience:",FULL_PATIENCE,flush=True)
print("Outputs:",FULL_ROOT,flush=True)

## 10 — Execute full LOSO with independent cache and checkpoint per fold

**Run once.** Each fold computes the training-only normalization reference, builds or reuses its own fingerprinted full-crown patch cache, trains or resumes M0 and M1, and evaluates the entire held-out forest using the existing immutable crown annotations. Completion is streamed after every fold. The pilot is never overwritten.

**Interpretation:** M0/M1 use native RGB for held-out testing (no target-domain normalization); the fold reference is recorded for reproducibility but unused by M0/M1. No SAM and no Luki in this workflow. The held-out labels are accessed **only after training** for evaluation, not to select the best epoch.

In [ ]:
def fold_cache04():
    # All eligible crowns, including natural species prevalences; split buffers excluded.
    global X,Y,S,SI,CI,NPZ
    if NPZ.exists() and not FORCE_REBUILD_CACHE:
        with np.load(NPZ,allow_pickle=False) as obj:
            X,Y,S,SI,CI=[obj[k] for k in ["X","Y","S","SI","CI"]]
        print("CACHED full-crown patches",len(X),NPZ.name,flush=True)
    else:
        X,Y,S,SI,CI=generate_cache03()
    assert set(SI)==set(training_sites)
    assert (S==0).any() and (S==1).any()
    return len(X)

def evaluate_fold04(variant):
    ck=RUN_OUT/variant/"best_model.pt"
    assert ck.exists(),f"Missing selected checkpoint: {ck}"
    seed_all(SEED)
    model=new_model04()
    state=torch.load(ck,map_location=DEVICE,weights_only=False)
    assert state["run_id"]==RUN_ID and state["variant"]==variant
    model.load_state_dict(state["model"])
    pred=test_crowns04(model,variant,TEST_CROWNS)
    del model
    if DEVICE=="cuda":torch.cuda.empty_cache()
    gc.collect()
    assert len(pred)>0 and pred.label.nunique()==2
    assert pred.crown_id.is_unique
    assert set(pred.crown_id).issubset(set(TEST_CROWNS.crown_id))
    pred.to_csv(RUN_OUT/variant/"full_heldout_crown_predictions.csv",index=False)
    return dict(fold=HELDOUT,variant=variant,
                reference_crowns=int(len(TEST_CROWNS)),
                evaluated_crowns=int(len(pred)),
                **summarize04(pred))

def run_full_fold04(holdout):
    global HELDOUT,training_sites,reference,GAIN,RUN_ID,RUN_OUT,NPZ
    global pilot_indices,TEST_CROWNS,RUN_VARIANTS,MAX_EPOCHS,PATIENCE,BATCH_SIZE
    assert holdout in SITES
    HELDOUT=holdout
    training_sites=[s for s in SITES if s!=HELDOUT]
    assert len(training_sites)==3
    # This reference uses ONLY the three development forests of this fold.
    reference=float(np.median([LUMINANCE_MEDIAN[s] for s in training_sites]))
    GAIN={s:float(np.clip(reference/max(LUMINANCE_MEDIAN[s],1e-6),
                           *GLOBAL_GAIN_BOUNDS)) for s in SITES}
    pilot_indices={(s,sp):ANN[s][ANN[s].split==sp].copy()
                   for s in training_sites for sp in ("training","validation")}
    TEST_CROWNS=ANN[HELDOUT].copy()  # ALL annotated crowns; original prevalence
    RUN_VARIANTS=FULL_VARIANTS.copy()
    MAX_EPOCHS=FULL_MAX_EPOCHS
    PATIENCE=FULL_PATIENCE
    BATCH_SIZE=FULL_BATCH_SIZE
    configuration=dict(stage="V2_04_full_loso_crown_patches_v1",
      fold=HELDOUT,train_sites=training_sites,
      train_ids={s:sorted(pilot_indices[s,"training"].crown_id.astype(str).tolist())
                 for s in training_sites},
      validation_ids={s:sorted(pilot_indices[s,"validation"].crown_id.astype(str).tolist())
                      for s in training_sites},
      patch=PATCH,seed=SEED,
      source={s:SOURCE_FINGERPRINT[s] for s in training_sites},
      photo_aug_version=1)
    RUN_ID=hashlib.sha256(json.dumps(configuration,sort_keys=True).encode()).hexdigest()[:12]
    RUN_OUT=FULL_ROOT/f"{HELDOUT}_{RUN_ID}"
    RUN_OUT.mkdir(exist_ok=True)
    NPZ=CACHE/f"full_{HELDOUT}_{RUN_ID}.npz"
    # A complete fold is resumable independently, including if Colab disconnects.
    summary_path=RUN_OUT/"04_fold_results.csv"
    if summary_path.exists():
        existing=pd.read_csv(summary_path)
        if set(existing.variant)==set(FULL_VARIANTS) and len(existing)==len(FULL_VARIANTS):
            print("REUSING COMPLETE FULL LOSO FOLD",HELDOUT,flush=True)
            return existing
    n=fold_cache04()
    print(f"\nFULL FOLD {HELDOUT}: train={int((S==0).sum())}, "
          f"val={int((S==1).sum())}, test={len(TEST_CROWNS)}; cached {n}",flush=True)
    results=[]
    for variant in FULL_VARIANTS:
        print("TRAIN:",HELDOUT,variant,flush=True)
        selected=train_variant04(variant)
        print("TEST:",HELDOUT,variant,flush=True)
        scores=evaluate_fold04(variant)
        scores["best_training_validation_ap"]=selected["best_validation_ap"]
        scores["run_id"]=RUN_ID
        results.append(scores)
        # Write cumulative results after every model, even if later folds fail.
        pd.DataFrame(results).to_csv(summary_path,index=False)
        print("DONE",HELDOUT,variant,
              "crown AUC",round(scores["auc"],4),
              "AP",round(scores["average_precision"],4),
              "F1@0.5",round(scores["f1"],4),flush=True)
    return pd.DataFrame(results)

ALL_FULL04=[]
if RUN_FULL_LOSO:
    assert DEVICE=="cuda","Choose a GPU runtime for full LOSO."
    for fold in FULL_FOLDS:
        print("\n"+"="*55+"\nFULL LOSO HELD-OUT:",fold,flush=True)
        df=run_full_fold04(fold)
        ALL_FULL04.append(df)
        pd.concat(ALL_FULL04,ignore_index=True).to_csv(FULL_SUMMARY,index=False)
    print("\nAll four folds finished. Saved:",FULL_SUMMARY,flush=True)
else:
    print("Full LOSO disabled. Set RUN_FULL_LOSO=True and rerun section 10.")

## 11 — Aggregate ROC-AUC / AP, calibration-sensitive metrics and versioned completion manifest

Report crown-level ROC-AUC and AP as **primary**, alongside threshold-dependent precision, recall, F1 and balanced accuracy. Macro averaging across forests avoids treating the largest forest as automatically most important. Flag missing predictions; preserve per-crown outputs for later absolute RGB brightness joins. **No claim of external Luki accuracy here.**

In [ ]:
if FULL_SUMMARY.exists():
    result=pd.read_csv(FULL_SUMMARY)
    assert set(result.fold)==set(FULL_FOLDS),(
        "Full LOSO incomplete; existing folds:",sorted(result.fold.unique()))
    assert len(result)==len(FULL_FOLDS)*len(FULL_VARIANTS)
    order=["fold","variant","reference_crowns","evaluated_crowns",
           "auc","average_precision","precision","recall","balanced_accuracy","f1",
           "false_positives","false_negatives","best_training_validation_ap"]
    display(result[order].round(4))
    macro=result.groupby("variant",as_index=False).agg(
        folds=("fold","nunique"),
        macro_crown_auc=("auc","mean"),
        macro_crown_ap=("average_precision","mean"),
        macro_f1_at_0p5=("f1","mean"),
        macro_balanced_accuracy_at_0p5=("balanced_accuracy","mean"),
        total_crowns=("evaluated_crowns","sum"),
        missing_crowns=("reference_crowns",lambda x:0)) # recomputed below
    missing=result.groupby("variant").apply(
        lambda d:int((d.reference_crowns-d.evaluated_crowns).sum()),
        include_groups=False)
    macro["missing_crowns"]=macro.variant.map(missing)
    macro.to_csv(FULL_ROOT/"04_full_LOSO_macro_summary.csv",index=False)
    display(macro.round(4))
    contract=dict(stage="Musanga V2/04 complete LOSO using manual crowns",
      folds=FULL_FOLDS,variants=FULL_VARIANTS,heldout_Luki=False,
      patch_size_px=PATCH,
      trained_on="All eligible train split crowns in three forests per fold",
      validated_on="All eligible validation split crowns in three forests per fold",
      test_on="All eligible reference crowns from the fourth forest",
      reference_split_source=str(REF),
      primary_metrics=["heldout_crown_ROC_AUC","heldout_crown_average_precision"],
      secondary_metrics=["F1_at_probability_0.5","balanced_accuracy","precision","recall"],
      image_input="native UAV RGB only; no SAM, no spatial resampling",
      threshold_policy="fixed 0.5 is diagnostic, not chosen on held-out test",
      production_warning="Crown-centred patch pilot/training is NOT full orthomosaic pixel-level test",
      luki="held aside for future, versioned new dark-region annotations",
      full_metrics=str(FULL_SUMMARY),
      macro_metrics=str(FULL_ROOT/"04_full_LOSO_macro_summary.csv"))
    (FULL_ROOT/"04_full_LOSO_manifest.json").write_text(json.dumps(contract,indent=2))
    print("Full LOSO finished; all pilot outputs remain intact.",flush=True)
else:
    print("Section 10 has not yet produced full LOSO metrics.")
# Restore pilot state, so revisiting earlier sections does not cross-contaminate.
if "PILOT_GLOBALS04" in globals():
    globals().update(PILOT_GLOBALS04)
    print("Restored pilot configuration and arrays.")

## 12 — Fresh-runtime setup for the final four-site M1 model

**Start here in a NEW Colab GPU runtime.** This section independently mounts Drive, loads the immutable four-site Notebook 01 references and split registers, defines the original U-Net training and patch functions, and locks the versioned Luki reference. It does **not** execute the pilot or four LOSO folds. After this cell, run Sections **13 and 14 only**.

The final training cache is separately fingerprinted and resumable; completed pilot and LOSO checkpoints remain untouched. No GPU is needed until Section 13, but select a GPU runtime before starting.

In [3]:
# FRESH-RUNTIME BOOTSTRAP: only the essential earlier DEFINITIONS are used.
# No pilot inference or four-fold LOSO is launched by this section.
%pip -q install rasterio geopandas pyogrio segmentation-models-pytorch scikit-learn tqdm matplotlib
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,rasterio
from rasterio.windows import Window
from rasterio.features import rasterize,geometry_mask
from shapely.geometry import box
from sklearn.metrics import roc_auc_score,average_precision_score,precision_score,recall_score,f1_score,balanced_accuracy_score
from tqdm.auto import tqdm
from IPython.display import display
import segmentation_models_pytorch as smp
import torch,torch.nn as nn
from torch.utils.data import Dataset,DataLoader
import json,hashlib,random,time,math,gc,sys
from google.colab import drive
MY=Path("/content/drive/MyDrive")
if not MY.exists():drive.mount("/content/drive")
ROOT=MY/"Colab Notebooks"/"Musanga"
GEO=ROOT/"data"/"Geo_Ref";AD=GEO/"Annotations"
V2=ROOT/"KVO_work"/"V2"
REF=V2/"01_reference_splits"
HARM=V2/"03_illumination_harmonization"
OUT=V2/"04_illumination_robust_training"
OUT.mkdir(parents=True,exist_ok=True)
CACHE=OUT/"pilot_patch_cache";CACHE.mkdir(exist_ok=True)
SITES={
 "MOS_1_2":("MOS_1_2_kvo.tif","Tree_Annotations_MOS_1_2_kvo.shp"),
 "Yoko":("Yoko_kvo_georef.tif","Tree_Annotations_Yoko_kvo_georef.shp"),
 "Yangambi":("Yangambi_georef.tif","Tree_Annotation_Yangambi_georef.shp"),
 "Baego":("Baego_georef.tif","Tree_Annotation_Baego.shp")}
# PILOT defaults: change only after inspecting first comparisons.
HELDOUT="Baego"
PILOT_TRAIN_CROWNS_PER_SITE=240  # cap across classes, balanced sampling
PILOT_VAL_CROWNS_PER_SITE=90
PILOT_TEST_CROWNS=180
PATCH=256
MAX_EPOCHS=4
PATIENCE=3
BATCH_SIZE=12
SEED=20261013
LR=2e-4
NUM_WORKERS=0  # Colab-safe; on-demand augmentation cheap, avoids multiprocessing state problems
IGNORE=255
MIN_AREA_M2=5.0
RUN_VARIANTS=["M0_native_historical","M1_native_photometric","M2_sitegain_photometric"]
FORCE_REBUILD_CACHE=False
FORCE_RESTART=False
RUN_TRAINING=False
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
assert HELDOUT in SITES
assert (REF/"01_spatial_split_summary.csv").exists(), "Run V2 Notebook 01 first"
assert (HARM/"03_site_luminance_and_fixed_gains.csv").exists(), "Run V2 Notebook 03 Section 02"
if DEVICE=="cpu":
    print("WARNING: CPU detected — select a GPU runtime before training.")
else: print("GPU:",torch.cuda.get_device_name(0))
print("PILOT heldout:",HELDOUT,"epochs:",MAX_EPOCHS,"output:",OUT)
def sha_file(path):
    stat=path.stat()
    return dict(path=str(path),size_bytes=stat.st_size,mtime_ns=stat.st_mtime_ns)
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
seed_all(SEED)

ANN={};SOURCE_FINGERPRINT={}
for site,(filename,shp) in SITES.items():
    rp=GEO/filename;ap=AD/shp
    assert rp.exists() and ap.exists(),site
    with rasterio.open(rp) as src:
        crs=src.crs;bounds=box(*src.bounds)
        assert crs and crs.is_projected
    g=gpd.read_file(ap)
    assert g.crs is not None and "Class_ID" in g.columns
    if g.crs!=crs:g=g.to_crs(crs)
    valid=g.geometry.notna()&(~g.geometry.is_empty)&g.geometry.is_valid
    g=g.loc[valid].copy()
    numerical=pd.to_numeric(g.Class_ID,errors="coerce")
    g=g.loc[numerical.notna()].copy()
    g["class_id"]=numerical.loc[g.index].astype(int)
    g["label"]=(g.class_id==1).astype(np.uint8)
    g["area_m2"]=g.geometry.area
    g=g.loc[(g.area_m2>=MIN_AREA_M2)&g.geometry.intersects(bounds)].copy()
    g["crown_id"]=site+":"+g.index.astype(str)
    reg=pd.read_csv(REF/f"01_{site}_split_register.csv",dtype={"crown_id":"string"})
    assert reg.crown_id.is_unique
    assert set(g.crown_id)==set(reg.crown_id),f"{site} annotation differs from locked Notebook 01 split"
    g=g.merge(reg[["crown_id","split","spatial_block"]],on="crown_id",how="left",validate="one_to_one")
    assert g.split.isin(["training","validation","buffered"]).all()
    ANN[site]=g
    SOURCE_FINGERPRINT[site]=dict(image=sha_file(rp),annotation=sha_file(ap),
                                   split=sha_file(REF/f"01_{site}_split_register.csv"))
    print(site,":",len(g),"crowns; train",sum(g.split=="training"),
          "val",sum(g.split=="validation"),"buffered",sum(g.split=="buffered"))
print("LOSO TEST:",HELDOUT,"uses all annotation classes but only a bounded pilot subsample.")

def centered_window(g,patch,src):
    cx,cy=g.centroid.x,g.centroid.y
    row,col=rasterio.transform.rowcol(src.transform,cx,cy)
    return Window(int(col)-patch//2,int(row)-patch//2,patch,patch)

def generate_cache03():
    xx=[];yy=[];ss=[];sites=[];ids=[]
    # A crop is selected using a crown centroid, but loss includes only the anchor
    # crown's own pixels. This avoids assigning a neighboring split's labels.
    for site in training_sites:
        with rasterio.open(GEO/SITES[site][0]) as src:
            for split_code,split in ((0,"training"),(1,"validation")):
                g=pilot_indices[site,split]
                for _,rec in tqdm(g.iterrows(),total=len(g),desc=f"{site}/{split}",leave=False):
                    w=centered_window(rec.geometry,PATCH,src)
                    rgb=src.read([1,2,3],window=w,boundless=True,fill_value=0)
                    arr=np.moveaxis(rgb,0,-1).astype(np.uint8)
                    original=arr
                    valid=~((original==0).all(axis=2)|
                            (original==255).all(axis=2)|
                            ((original[:,:,0]==0)&(original[:,:,1]==0)&(original[:,:,2]==255)))
                    mask=rasterize([(rec.geometry,int(rec.label))],out_shape=(PATCH,PATCH),
                                   transform=src.window_transform(w),fill=IGNORE,dtype="uint8")
                    mask[~valid]=IGNORE
                    if int((mask!=IGNORE).sum())<12:continue
                    xx.append(arr);yy.append(mask);ss.append(split_code)
                    sites.append(site);ids.append(str(rec.crown_id))
    assert xx and len(set(ss))==2
    X=np.stack(xx);Y=np.stack(yy);S=np.array(ss,np.uint8)
    SI=np.array(sites);CI=np.array(ids)
    print("Caching",len(X),"patches:",round(X.nbytes/1e6,1),"MB raw RGB",flush=True)
    np.savez_compressed(NPZ,X=X,Y=Y,S=S,SI=SI,CI=CI)
    return X,Y,S,SI,CI

MEAN=np.array([.485,.456,.406],dtype=np.float32)[None,None,:]
STD=np.array([.229,.224,.225],dtype=np.float32)[None,None,:]
def shadow04(x,rng):
    # Spatially broad and smooth illumination change on the crop.
    # Default local mask variation is differentiable and multiplicative, not recoloring.
    h,w=x.shape[:2]
    yy,xx=np.mgrid[:h,:w].astype(np.float32)
    theta=rng.uniform(0,np.pi)
    normal=(np.cos(theta)*(xx-w/2)+np.sin(theta)*(yy-h/2))
    offset=rng.uniform(-.35,.35)*max(h,w)
    edge_width=rng.uniform(20,90)
    transition=np.clip((normal-offset)/edge_width,-12,12)
    weights=1/(1+np.exp(-transition))
    darkness=rng.uniform(.45,.90)
    return np.clip(x*(darkness+(1-darkness)*weights[:,:,None]),0,1)
class PilotDataset04(Dataset):
    def __init__(self,which,variant,epoch=0):
        self.indices=np.flatnonzero(S==which)
        self.variant=variant;self.epoch=epoch;self.training=(which==0)
    def __len__(self):return len(self.indices)
    def __getitem__(self,j):
        index=int(self.indices[j])
        img=X[index].astype(np.float32)/255.
        mask=Y[index].copy()
        site=str(SI[index])
        if self.variant=="M2_sitegain_photometric":
            img=np.clip(img*GAIN[site],0,1)
        if self.training:
            rng=np.random.default_rng(SEED+self.epoch*1_000_003+index*101)
            if rng.random()<.5:img=img[:,::-1,:].copy();mask=mask[:,::-1].copy()
            if rng.random()<.5:img=img[::-1,:,:].copy();mask=mask[::-1,:].copy()
            k=int(rng.integers(0,4))
            img=np.rot90(img,k).copy();mask=np.rot90(mask,k).copy()
            if self.variant=="M0_native_historical":
                img=np.clip(img*rng.uniform(.90,1.10),0,1)
            else:
                img=np.clip(img*rng.uniform(.70,1.30),0,1)
                img=np.clip((img-.5)*rng.uniform(.80,1.20)+.5,0,1)
                img=np.power(img,float(rng.uniform(.75,1.35)))
                img=np.clip(img*rng.uniform(.92,1.08,size=(1,1,3)),0,1)
                if rng.random()<.55:img=shadow04(img,rng)
        valid=mask!=IGNORE
        if valid.sum()==0:raise RuntimeError("Empty training patch after source QA")
        tensor=np.moveaxis(((img-MEAN)/STD).astype(np.float32),-1,0).copy()
        return torch.from_numpy(tensor),torch.from_numpy(mask.astype(np.int64))
def new_model04():
    return smp.Unet(encoder_name="resnet18",encoder_weights="imagenet",
                    in_channels=3,classes=1).to(DEVICE)
def make_loader04(code,variant,epoch):
    ds=PilotDataset04(code,variant,epoch)
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=(code==0),
        pin_memory=DEVICE=="cuda",num_workers=NUM_WORKERS)
print("Each variant has identical architecture, patch inventory and RNG seed.")


def score_valid04(model,loader):
    model.eval();true=[];pred=[]
    with torch.inference_mode():
        for x,y in loader:
            x=x.to(DEVICE,non_blocking=True)
            with torch.amp.autocast("cuda",enabled=DEVICE=="cuda"):
                out=torch.sigmoid(model(x).squeeze(1))
            mask=(y!=IGNORE)
            true.append(y[mask].numpy().astype(np.uint8))
            pred.append(out.detach().cpu()[mask].numpy().astype(np.float32))
    target=np.concatenate(true);prob=np.concatenate(pred)
    return dict(ap=float(average_precision_score(target,prob)) if target.sum() else np.nan,
        auc=float(roc_auc_score(target,prob)) if len(np.unique(target))==2 else np.nan,
        pixels=int(len(target)))
def train_variant04(variant):
    folder=RUN_OUT/variant;folder.mkdir(parents=True,exist_ok=True)
    progress=folder/"last_training_state.pt";bestfile=folder/"best_model.pt"
    history_file=folder/"epoch_history.csv";marker=folder/"pilot_complete.json"
    if marker.exists() and bestfile.exists() and not FORCE_RESTART:
        print("REUSING COMPLETED PILOT:",variant,flush=True)
        return json.loads(marker.read_text())
    seed_all(SEED)
    model=new_model04()
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
    scaler=torch.amp.GradScaler("cuda",enabled=DEVICE=="cuda")
    history=[];start=1;best=-1.;wait=0
    if progress.exists() and not FORCE_RESTART:
        state=torch.load(progress,map_location=DEVICE,weights_only=False)
        assert state["variant"]==variant and state["run_id"]==RUN_ID
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        best=float(state["best"]);wait=int(state["wait"])
        start=int(state["epoch"])+1
        history=state["history"]
        print("RESUMING:",variant,"epoch",start,"best AP",round(best,4),flush=True)
    for epoch in range(start,MAX_EPOCHS+1):
        begin=time.monotonic();model.train();losses=[]
        loader=make_loader04(0,variant,epoch)
        for bi,(x,y) in enumerate(loader,1):
            x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda",enabled=DEVICE=="cuda"):
                logits=model(x).squeeze(1)
                valid=(y!=IGNORE)
                loss=nn.functional.binary_cross_entropy_with_logits(logits[valid],y[valid].float())
            scaler.scale(loss).backward()
            scaler.step(optimizer);scaler.update()
            losses.append(float(loss.item()))
            if bi==1 or bi%25==0 or bi==len(loader):
                print(f"  {variant} epoch {epoch} batch {bi}/{len(loader)} loss {np.mean(losses):.4f}",
                      flush=True)
        validation=score_valid04(model,make_loader04(1,variant,0))
        improved=np.isfinite(validation["ap"]) and validation["ap"]>best
        if improved:
            best=validation["ap"];wait=0
            torch.save(dict(model=model.state_dict(),epoch=epoch,variant=variant,
                            run_id=RUN_ID,best_validation_ap=best),bestfile)
        else:wait+=1
        info=dict(epoch=epoch,train_loss=float(np.mean(losses)),validation_ap=validation["ap"],
                  validation_auc=validation["auc"],validation_pixels=validation["pixels"],
                  seconds=float(time.monotonic()-begin),best_validation_ap=best,wait=wait)
        history.append(info)
        pd.DataFrame(history).to_csv(history_file,index=False)
        torch.save(dict(variant=variant,run_id=RUN_ID,epoch=epoch,
            model=model.state_dict(),optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),
            best=best,wait=wait,history=history),progress)
        print(f"PROGRESS {variant} epoch {epoch}/{MAX_EPOCHS}: loss={info['train_loss']:.4f}, "
              f"AP={info['validation_ap']:.4f}, AUC={info['validation_auc']:.4f}, "
              f"best_AP={best:.4f}, {info['seconds']:.1f}s",flush=True)
        if wait>=PATIENCE:
            print("EARLY STOP:",variant,"after",epoch,"epochs",flush=True);break
    if not bestfile.exists():raise RuntimeError(f"Best checkpoint never saved for {variant}")
    summary=dict(variant=variant,run_id=RUN_ID,heldout=HELDOUT,
                 epochs_completed=len(history),best_validation_ap=best,
                 selection_basis="spatial validation pixels on three training forests",
                 heldout_labels_used_for_selection=False)
    marker.write_text(json.dumps(summary,indent=2))
    del model,optimizer;gc.collect()
    if DEVICE=="cuda":torch.cuda.empty_cache()
    return summary

FULL_BATCH_SIZE=12

# --------------------- FINAL-MODEL CONFIGURATION ---------------------
FINAL04=OUT/"final_four_site_M1";FINAL04.mkdir(parents=True,exist_ok=True)
FINAL_VARIANT04="M1_native_photometric"
FINAL_MAX_EPOCHS04=20
FINAL_PATIENCE04=5
RUN_FINAL_M1_04=True
RUN_LUKI_04=True
LUKI_RGB04=GEO/"Luki.tif"
LUKI_META04=REF/"01_luki_current_reference_version.json"
assert LUKI_META04.exists(),"First run updated Notebook 01 with the new Luki annotations"
LUKI_MANIFEST04=json.loads(LUKI_META04.read_text())
LUKI_VERSION04=LUKI_MANIFEST04["reference_id"]
LUKI_REFERENCE04=Path(LUKI_MANIFEST04["frozen_copy"])
assert LUKI_REFERENCE04.exists(),"Missing frozen external Luki version"
LUKI_CROWNS04=gpd.read_file(LUKI_REFERENCE04,layer="external_evaluation")
assert len(LUKI_CROWNS04)==int(LUKI_MANIFEST04["crowns"])
assert int(LUKI_CROWNS04.label.sum())==int(LUKI_MANIFEST04["musanga"])
assert LUKI_CROWNS04.crown_id.is_unique
assert LUKI_CROWNS04.crs is not None
with rasterio.open(LUKI_RGB04) as src:
    assert LUKI_CROWNS04.crs==src.crs
print("Final M1 source forests",list(SITES))
print("Luki independent source",LUKI_VERSION04,
      len(LUKI_CROWNS04),"crowns (reserved until model fully frozen)")
print("Final 4-site M1 directory:",FINAL04)


GPU: NVIDIA L4
PILOT heldout: Baego epochs: 4 output: /content/drive/MyDrive/Colab Notebooks/Musanga/KVO_work/V2/04_illumination_robust_training
MOS_1_2 : 1132 crowns; train 863 val 240 buffered 29
Yoko : 1792 crowns; train 1215 val 417 buffered 160
Yangambi : 944 crowns; train 735 val 183 buffered 26
Baego : 749 crowns; train 534 val 198 buffered 17
LOSO TEST: Baego uses all annotation classes but only a bounded pilot subsample.
Each variant has identical architecture, patch inventory and RNG seed.
Final M1 source forests ['MOS_1_2', 'Yoko', 'Yangambi', 'Baego']
Luki independent source d957b570055bf5dd 479 crowns (reserved until model fully frozen)
Final 4-site M1 directory: /content/drive/MyDrive/Colab Notebooks/Musanga/KVO_work/V2/04_illumination_robust_training/final_four_site_M1


## 13 — Final M1 training on all four original forests (resumable)

Run directly after Section 12 in a fresh GPU runtime. Read or build one four-forest native-RGB crown-centred cache using the frozen Notebook 01 training/validation splits. Train **M1 only** with the previously validated photometric recipe; save epoch checkpoints, select the model using development-forest validation pixel AP, and never consult Luki labels. No preceding pilot or LOSO cells are required.

In [4]:
FINAL_CONFIG04=dict(stage="four_site_M1_v1",seed=SEED,patch=PATCH,
  training_sites=list(SITES),train_crown_ids={
      site:sorted(ANN[site].loc[ANN[site].split=="training","crown_id"].astype(str).tolist())
      for site in SITES},
  val_crown_ids={
      site:sorted(ANN[site].loc[ANN[site].split=="validation","crown_id"].astype(str).tolist())
      for site in SITES},
  source_fingerprints=SOURCE_FINGERPRINT,
  photo_augmentation="M1_native_photometric_same_as_LOSO",
  max_epochs=FINAL_MAX_EPOCHS04,patience=FINAL_PATIENCE04)
FINAL_RUN_ID04=hashlib.sha256(json.dumps(FINAL_CONFIG04,sort_keys=True).encode()).hexdigest()[:12]
FINAL_RUN04=FINAL04/FINAL_RUN_ID04
FINAL_RUN04.mkdir(parents=True,exist_ok=True)
FINAL_CACHE04=CACHE/f"final_four_site_M1_{FINAL_RUN_ID04}.npz"

def final_prepare04():
    global HELDOUT,training_sites,reference,GAIN,RUN_ID,RUN_OUT,NPZ
    global pilot_indices,TEST_CROWNS,X,Y,S,SI,CI
    global RUN_VARIANTS,MAX_EPOCHS,PATIENCE,BATCH_SIZE,FORCE_RESTART
    # No fifth held-out development site: change training_sites to ALL four.
    HELDOUT="FINAL_FOUR_SITE_M1"
    training_sites=list(SITES)
    reference=np.nan # M1 uses native RGB: no site reference or gain is required
    GAIN={s:1.0 for s in SITES} # M1 never uses site gains
    pilot_indices={(site,sp):ANN[site][ANN[site].split==sp].copy()
                   for site in training_sites for sp in ("training","validation")}
    TEST_CROWNS=gpd.GeoDataFrame()
    RUN_ID=FINAL_RUN_ID04
    RUN_OUT=FINAL_RUN04
    NPZ=FINAL_CACHE04
    RUN_VARIANTS=[FINAL_VARIANT04]
    MAX_EPOCHS=FINAL_MAX_EPOCHS04
    PATIENCE=FINAL_PATIENCE04
    BATCH_SIZE=FULL_BATCH_SIZE
    FORCE_RESTART=False
    if NPZ.exists():
        with np.load(NPZ,allow_pickle=False) as cache:
            X,Y,S,SI,CI=[cache[k] for k in ["X","Y","S","SI","CI"]]
        print("REUSED full four-site training cache:",len(X),flush=True)
    else:
        X,Y,S,SI,CI=generate_cache03()
    assert set(SI)==set(SITES)
    assert (S==0).any() and (S==1).any()
    print("Four-site data:",len(X),"patches",
          int((S==0).sum()),"training",int((S==1).sum()),"validation",flush=True)
    print("Run ID:",RUN_ID,flush=True)

FINAL_CKPT04=FINAL_RUN04/FINAL_VARIANT04/"best_model.pt"
if RUN_FINAL_M1_04:
    assert DEVICE=="cuda","Select a Colab GPU runtime."
    final_prepare04()
    FINAL_TRAIN04=train_variant04(FINAL_VARIANT04)
    assert FINAL_CKPT04.exists()
    (FINAL_RUN04/"04_final_model_contract.json").write_text(json.dumps(dict(
        run_id=FINAL_RUN_ID04,architecture="smp.Unet(resnet18,ImageNet encoder)",
        model=FINAL_VARIANT04,source_forests=list(SITES),
        best_checkpoint=str(FINAL_CKPT04),
        spatial_training_and_validation="Notebook 01 250m blocks/30m buffer",
        class_labels="Class_ID=1 Musanga, other numeric Class_ID non-Musanga",
        best_validation_ap=FINAL_TRAIN04["best_validation_ap"],
        epochs_completed=FINAL_TRAIN04["epochs_completed"],
        sampling="256px native GSD crown-centred single annotated crown pixel loss",
        validation="spatial validation pixels across all four forests",
        luki="Not seen during training, validation, checkpoint selection or hyperparameter tuning",
        source_fingerprints=SOURCE_FINGERPRINT,
        limitation="Not wall-to-wall probability map; spatial seams and shadow patches require further QA"
    ),indent=2))
    print("FINAL FOUR-SITE M1 READY:",FINAL_CKPT04,flush=True)

else:
    print("Training disabled; expecting an existing final checkpoint:",FINAL_CKPT04)


MOS_1_2/training:   0%|          | 0/863 [00:00<?, ?it/s]

MOS_1_2/validation:   0%|          | 0/240 [00:00<?, ?it/s]

Yoko/training:   0%|          | 0/1215 [00:00<?, ?it/s]

Yoko/validation:   0%|          | 0/417 [00:00<?, ?it/s]

Yangambi/training:   0%|          | 0/735 [00:00<?, ?it/s]

...

Caching 4385 patches: 862.1 MB raw RGB
Four-site data: 4385 patches 3347 training 1038 validation
Run ID: 337073e7b74f


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 46.8MB            

model.safetensors: downloading bytes:           |  0.00B            

  M1_native_photometric epoch 1 batch 1/279 loss 0.6495
  M1_native_photometric epoch 1 batch 25/279 loss 0.5369
  M1_native_photometric epoch 1 batch 50/279 loss 0.4473
  M1_native_photometric epoch 1 batch 75/279 loss 0.4086
  M1_native_photometric epoch 1 batch 100/279 loss 0.3871
  M1_native_photometric epoch 1 batch 125/279 loss 0.3785
  M1_native_photometric epoch 1 batch 150/279 loss 0.3613
  M1_native_photometric epoch 1 batch 175/279 loss 0.3470
  M1_native_photometric epoch 1 batch 200/279 loss 0.3484
  M1_native_photometric epoch 1 batch 225/279 loss 0.3404
  M1_native_photometric epoch 1 batch 250/279 loss 0.3323
  M1_native_photometric epoch 1 batch 275/279 loss 0.3221
  M1_native_photometric epoch 1 batch 279/279 loss 0.3231
PROGRESS M1_native_photometric epoch 1/20: loss=0.3231, AP=0.9704, AUC=0.9738, best_AP=0.9704, 46.4s
  M1_native_photometric epoch 2 batch 1/279 loss 0.2852
  M1_native_photometric epoch 2 batch 25/279 loss 0.2582
  M1_native_photometric epoch 2 batch

## 14 — Independent external Luki crown ROC-AUC/AP, plus illumination diagnostics

Evaluate **exactly the frozen reference version** printed in Section 12. For each manually delineated crown read a native RGB window centred on the crown, predict with the frozen M1 model, and average probability over its rasterized geometry. Reject crowns whose polygon cannot fit inside the inference window or lack sufficient valid RGB support rather than silently biasing crown statistics. **ROC-AUC and AP remain primary**; F1 at 0.5 is secondary. Also join the actual Luki native-RGB luminance from Notebook 01, and summarize errors by **common 0–1 absolute brightness bins** and **Luki-relative quartiles**.

The crown diagnostic does **not** test broad unannotated shadow areas, ortho seams or model behaviour on full probability maps. Later we will generate those maps and test their suitability for satellite targets.

In [5]:
def predict_luki_crowns04(model,gdf):
    rows=[]
    model.eval()
    with rasterio.open(LUKI_RGB04) as src:
        for _,rec in tqdm(gdf.iterrows(),total=len(gdf),desc="Final M1 external Luki"):
            win=centered_window(rec.geometry,PATCH,src)
            rgb=src.read([1,2,3],window=win,boundless=True,fill_value=0)
            raw=np.moveaxis(rgb,0,-1).astype(np.uint8)
            allowed=~((raw==0).all(axis=2)|(raw==255).all(axis=2)|
                      ((raw[:,:,0]==0)&(raw[:,:,1]==0)&(raw[:,:,2]==255)))
            inside=geometry_mask([rec.geometry],out_shape=(PATCH,PATCH),
                transform=src.window_transform(win),invert=True)
            # Reject truncated crowns so the inference subset matches reference geometry.
            expected=max(1.,float(rec.geometry.area)/abs(src.transform.a*src.transform.e))
            coverage=float(inside.sum()/expected)
            mask=inside&allowed
            base=dict(site="Luki",crown_id=str(rec.crown_id),label=int(rec.label),
                valid_pixels=int(mask.sum()),estimated_geometry_coverage=coverage)
            if coverage<.90 or mask.sum()<10:
                rows.append(dict(**base,score=np.nan,qa="truncated_or_low_support"))
                continue
            img=raw.astype(np.float32)/255.
            arr=np.moveaxis(((img-MEAN)/STD).astype(np.float32),-1,0).copy()
            with torch.inference_mode(),torch.amp.autocast("cuda",enabled=DEVICE=="cuda"):
                p=torch.sigmoid(model(torch.from_numpy(arr[None]).to(DEVICE))).squeeze().float().cpu().numpy()
            rows.append(dict(**base,score=float(np.mean(p[mask])),qa="ok"))
    return pd.DataFrame(rows)

LUKI_EVAL_DIR04=FINAL_RUN04/"luki_external"/LUKI_VERSION04
LUKI_EVAL_DIR04.mkdir(parents=True,exist_ok=True)
RESULT_PATH04=LUKI_EVAL_DIR04/"04_final_M1_luki_crown_predictions.csv"
if RUN_LUKI_04:
    assert FINAL_CKPT04.exists(),"Run Section 13 to train the 4-site final model first."
    if RESULT_PATH04.exists():
        preds=pd.read_csv(RESULT_PATH04,dtype={"crown_id":"string"})
        print("REUSED version-locked external predictions",RESULT_PATH04)
    else:
        model=new_model04()
        state=torch.load(FINAL_CKPT04,map_location=DEVICE,weights_only=False)
        assert state["run_id"]==FINAL_RUN_ID04
        model.load_state_dict(state["model"])
        preds=predict_luki_crowns04(model,LUKI_CROWNS04)
        preds.to_csv(RESULT_PATH04,index=False)
        del model
        if DEVICE=="cuda":torch.cuda.empty_cache()
    assert len(preds)==len(LUKI_CROWNS04)
    assert preds.crown_id.is_unique
    light=pd.read_csv(REF/"01_crown_illumination.csv",dtype={"crown_id":"string"})
    luki_light=light.loc[light.site=="Luki",["crown_id","median_luminance"]]
    assert luki_light.crown_id.is_unique
    preds=preds.merge(luki_light,on="crown_id",how="left",validate="one_to_one")
    assert preds.median_luminance.notna().all()
    preds["fixed_RGB_bin"]=pd.cut(preds.median_luminance,bins=np.linspace(0,1,11),include_lowest=True)
    quartiles=preds.median_luminance.quantile([.25,.75])
    preds["relative_light"]=np.where(preds.median_luminance<quartiles.loc[.25],
        "darkest_25pct",np.where(preds.median_luminance>quartiles.loc[.75],
        "brightest_25pct","middle_50pct"))
    evaluated=preds[preds.qa=="ok"].copy()
    print("Luki reference crowns",len(preds),"with valid centre patch",len(evaluated),
          "excluded due to truncation/coverage",len(preds)-len(evaluated),flush=True)
    if evaluated.label.nunique()!=2:raise RuntimeError("Both species classes required for AUC/AP")
    p=evaluated.score.to_numpy(float);y=evaluated.label.to_numpy(int)
    q=(p>=.5).astype(int)
    metrics=dict(model="final_four_site_M1",reference_version=LUKI_VERSION04,
        reference_crowns=len(preds),evaluated_crowns=len(evaluated),
        roc_auc=float(roc_auc_score(y,p)),average_precision=float(average_precision_score(y,p)),
        precision_0p5=float(precision_score(y,q,zero_division=0)),
        recall_0p5=float(recall_score(y,q,zero_division=0)),
        f1_0p5=float(f1_score(y,q,zero_division=0)),
        balanced_accuracy_0p5=float(balanced_accuracy_score(y,q)),
        false_positives=int(((y==0)&(q==1)).sum()),
        false_negatives=int(((y==1)&(q==0)).sum()))
    pd.DataFrame([metrics]).to_csv(LUKI_EVAL_DIR04/"04_final_M1_luki_metrics.csv",index=False)
    display(pd.DataFrame([metrics]).round(4))
    details=[]
    for name,col in [("absolute_RGB_bin","fixed_RGB_bin"),("relative_brightness","relative_light")]:
        for value,gg in evaluated.groupby(col,observed=True):
            pos=gg[gg.label==1];neg=gg[gg.label==0]
            details.append(dict(stratum_type=name,stratum=str(value),n=len(gg),
                musanga=len(pos),other=len(neg),
                mean_probability=float(gg.score.mean()),
                musanga_recall=float((pos.score>=.5).mean()) if len(pos) else np.nan,
                other_false_positive_rate=float((neg.score>=.5).mean()) if len(neg) else np.nan,
                auc=float(roc_auc_score(gg.label,gg.score)) if gg.label.nunique()==2 else np.nan,
                ap=float(average_precision_score(gg.label,gg.score)) if gg.label.nunique()==2 else np.nan))
    strata=pd.DataFrame(details)
    strata.to_csv(LUKI_EVAL_DIR04/"04_final_M1_luki_brightness_metrics.csv",index=False)
    display(strata.round(4))
    preds.to_csv(LUKI_EVAL_DIR04/"04_final_M1_luki_crown_predictions_with_brightness.csv",index=False)
    print("NO wall-to-wall probability map has been generated here.",flush=True)
    print("NEXT: independently compare spatial shadow artefacts and satellite target transfer.",flush=True)
else:
    print("Luki evaluation disabled.")


Final M1 external Luki:   0%|          | 0/479 [00:00<?, ?it/s]

Luki reference crowns 479 with valid centre patch 406 excluded due to truncation/coverage 73


,model,reference_version,reference_crowns,evaluated_crowns,roc_auc,average_precision,precision_0p5,recall_0p5,f1_0p5,balanced_accuracy_0p5,false_positives,false_negatives
0,final_four_site_M1,d957b570055bf5dd,479,406,0.9845,0.9824,0.974,0.9414,0.9574,0.9527,6,14


,stratum_type,stratum,n,musanga,other,mean_probability,musanga_recall,other_false_positive_rate,auc,ap
0,absolute_RGB_bin,"(0.3, 0.4]",3,3,0,0.9415,1.0000,NaN,NaN,NaN
1,absolute_RGB_bin,"(0.4, 0.5]",47,44,3,0.8946,1.0000,0.3333,0.9924,0.9995
2,absolute_RGB_bin,"(0.5, 0.6]",173,138,35,0.7570,0.9638,0.0857,0.9681,0.9811
3,absolute_RGB_bin,"(0.6, 0.7]",137,46,91,0.2972,0.8478,0.0110,0.9859,0.9773
4,absolute_RGB_bin,"(0.7, 0.8]",46,8,38,0.1336,0.7500,0.0263,0.9342,0.8079
5,relative_brightness,brightest_25pct,101,19,82,0.1337,0.6316,0.0122,0.9499,0.8629
6,relative_brightness,darkest_25pct,109,98,11,0.8658,1.0000,0.2727,0.9972,0.9997
7,relative_brightness,middle_50pct,196,122,74,0.5858,0.9426,0.0270,0.9827,0.9745


NO wall-to-wall probability map has been generated here.
NEXT: independently compare spatial shadow artefacts and satellite target transfer.


## 15 — Luki: illumination–Musanga probability correlations, frozen U-Net versus M1

**Run after Section 14.** No training or new inference is needed. The comparison uses the **same Luki reference version and identical crown IDs**, joining the existing Notebook 02 frozen-U-Net predictions and Section 14 final M1 predictions. Spearman rank correlations between native RGB crown luminance and predicted Musanga probability are reported overall and **separately for Musanga and non-Musanga crowns**, with paired bootstrap confidence intervals.

The frozen model averaged predictions across entire polygons, whereas current M1 prediction is based on a 256-pixel centered patch; thus different spatial support remains a limitation even for matched crowns. The crucial later test is wall-to-wall probability maps, shadow patterns and satellite transfer; crown correlations alone cannot prove absence of illumination artefacts.

In [ ]:
# Section 15: matched-crown, species-conditional illumination sensitivity.
# Uses only stored outputs; no extra training or GPU required.
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
OLD15=V2/"02_frozen_baseline"/"by_luki_reference"/LUKI_VERSION04/"02_frozen_crown_predictions.csv"
NEW15=FINAL_RUN04/"luki_external"/LUKI_VERSION04/"04_final_M1_luki_crown_predictions_with_brightness.csv"
REPORT15=FINAL_RUN04/"luki_external"/LUKI_VERSION04/"illumination_comparison"
REPORT15.mkdir(parents=True,exist_ok=True)
assert OLD15.exists(),f"Notebook 02 frozen Luki predictions missing: {OLD15}"
assert NEW15.exists(),f"Section 14 final M1 Luki predictions missing: {NEW15}"
frozen15=pd.read_csv(OLD15,dtype={"crown_id":str})
frozen15=frozen15.loc[(frozen15.site=="Luki")&(frozen15.qa=="ok"),
                       ["crown_id","label","mean_p"]].rename(
                       columns={"label":"frozen_label","mean_p":"frozen_p"})
m115=pd.read_csv(NEW15,dtype={"crown_id":str})
m115=m115.loc[m115.qa=="ok",["crown_id","label","score","median_luminance"]].rename(
                      columns={"score":"M1_p"})
assert frozen15.crown_id.is_unique and m115.crown_id.is_unique
paired15=m115.merge(frozen15,on="crown_id",validate="one_to_one")
assert len(paired15)>0 and paired15.label.nunique()==2
assert (paired15.label==paired15.frozen_label).all(),"Label discrepancy between references"
assert paired15[["median_luminance","frozen_p","M1_p"]].notna().all().all()
print("Matching crown sample:",len(paired15),"of",len(LUKI_CROWNS04),
      "Luki crowns; Musanga",sum(paired15.label==1),"non-Musanga",sum(paired15.label==0))
rng15=np.random.default_rng(SEED+1500)
results15=[]
for name,gg in [("All (confounded)",paired15),
                ("Musanga only",paired15[paired15.label==1]),
                ("Non-Musanga only",paired15[paired15.label==0])]:
    lum=gg.median_luminance.to_numpy(float)
    original=gg.frozen_p.to_numpy(float)
    newer=gg.M1_p.to_numpy(float)
    oldrho=float(spearmanr(lum,original).statistic)
    newrho=float(spearmanr(lum,newer).statistic)
    samples=[]
    for k in range(1000):
        ix=rng15.integers(0,len(gg),len(gg))
        if len(np.unique(lum[ix]))<3:continue
        a=float(spearmanr(lum[ix],original[ix]).statistic)
        b=float(spearmanr(lum[ix],newer[ix]).statistic)
        if np.isfinite(a) and np.isfinite(b):
            samples.append([a,b,b-a,abs(b)-abs(a)])
    ci=np.quantile(np.asarray(samples),[.025,.975],axis=0)
    results15.append(dict(group=name,n=len(gg),
        frozen_rho=oldrho,M1_rho=newrho,
        signed_delta=newrho-oldrho,change_in_absolute_rho=abs(newrho)-abs(oldrho),
        frozen_95CI_low=ci[0,0],frozen_95CI_high=ci[1,0],
        M1_95CI_low=ci[0,1],M1_95CI_high=ci[1,1],
        absolute_change_95CI_low=ci[0,3],absolute_change_95CI_high=ci[1,3]))
correlations15=pd.DataFrame(results15)
correlations15.to_csv(REPORT15/"15_Luki_illumination_correlations.csv",index=False)
display(correlations15.round(3))

# Same matched crown sample, same luminance bins for both models.
paired15["brightness_bin"]=pd.cut(paired15.median_luminance,
    np.linspace(0,1,11),include_lowest=True)
rows15=[]
for band,gg in paired15.groupby("brightness_bin",observed=True):
    for model,col in [("Frozen","frozen_p"),("M1","M1_p")]:
        positives=gg[gg.label==1];negatives=gg[gg.label==0]
        rows15.append(dict(bin=str(band),model=model,crowns=len(gg),
            musanga=len(positives),non_musanga=len(negatives),
            median_luminance=float(gg.median_luminance.median()),
            mean_probability=float(gg[col].mean()),
            musanga_recall=float((positives[col]>=.5).mean()) if len(positives) else np.nan,
            non_musanga_false_positive_rate=float((negatives[col]>=.5).mean()) if len(negatives) else np.nan))
bins15=pd.DataFrame(rows15)
bins15.to_csv(REPORT15/"15_Luki_matched_brightness_bins.csv",index=False)
display(bins15.round(3))
fig,axes=plt.subplots(2,2,figsize=(12,9),sharex=True,sharey=True,constrained_layout=True)
for row,cls in enumerate([1,0]):
    group=paired15[paired15.label==cls]
    for col,(name,score) in enumerate([("Frozen production U-Net","frozen_p"),
                                        ("Final four-site M1","M1_p")]):
        ax=axes[row,col]
        ax.scatter(group.median_luminance,group[score],alpha=.4,s=16)
        rho=spearmanr(group.median_luminance,group[score]).statistic
        ax.set_title(f"{'Musanga' if cls else 'Non-Musanga'} — {name}\nn={len(group)}; Spearman ρ={rho:.3f}")
        ax.set(xlabel="Crown native-RGB luminance",ylabel="Predicted P(Musanga)",
               xlim=(0,1),ylim=(0,1))
        ax.axhline(.5,color="gray",ls="--",lw=1)
fig.savefig(REPORT15/"15_Luki_illumination_vs_probability.png",dpi=160)
plt.show()
paired15.to_csv(REPORT15/"15_Luki_matched_crown_probabilities.csv",index=False)
print("Section 15 complete:",REPORT15)
print("These are crown-level correlations, NOT proof that wall-to-wall mosaics lack shadow artefacts.")


## 16 — Final M1: full-resolution Luki Musanga probability map (GPU, resumable)

Produce a **native-grid, native-resolution** probability GeoTIFF from the already trained final four-site M1 checkpoint. The output is stored exclusively under `KVO_work/V2/04_illumination_robust_training/final_four_site_M1/<run_id>/luki_wall_to_wall/` and never overwrites an original UAV or historical production probability map.

**Efficient edge-safe inference:** each non-overlapping **512 × 512 output tile** is predicted from a **640 × 640 input window with a 64-pixel context halo** on all sides; the network's halo predictions are discarded. This avoids dividing a ~1.5-billion-pixel orthomosaic into large RAM arrays and minimizes tile-edge artefacts. The raster is written directly, one tile at a time, as `uint16` probabilities scaled 0–10000, with `65535` as no-data, matching the existing frozen probability-map convention. Pixel-validity rules use **RGB only** (exclude black 0/0/0, white 255/255/255, and pure blue 0/0/255); **never Luki's unreliable 4th band/alpha**. Zero illumination corrections are applied at inference: this is a true test of M1's photometric robustness.

Run **Section 12 first in this runtime** (for common model definitions/path variables). Section 13 is needed only when the final M1 checkpoint does not yet exist. Section 16 reuses an existing checkpoint and has row-wise progress/resume state on Drive. This processes the **entire Luki image**, so budget potentially considerable GPU time and ~3 GB for the final compressed/uncompressed output; a 3-row smoke-test option is provided. Do not compare satellite transfer until the completed map has passed illumination QA.

**Note:** halo tiling is not weighted overlap-blending; it discards edge predictions while retaining context. Inspect tile seams in the resulting map before treating it as a scientific product.

In [ ]:
# V2 / NB04 / Section 16: memory-bounded native Luki inference.
# Depends on Section 12 definitions, NOT previous pilot/LOSO execution.
import os, time, json, hashlib
from pathlib import Path
import numpy as np, rasterio, torch
import segmentation_models_pytorch as smp
from rasterio.windows import Window
from tqdm.auto import tqdm

assert torch.cuda.is_available(),"Use a GPU runtime for full-image prediction."
assert "FINAL_RUN04" in globals() and "FINAL_CKPT04" in globals(), \
    "Run Section 12, then Section 13 if the final M1 model has not been trained."
assert FINAL_CKPT04.exists(),f"Missing final M1 checkpoint: {FINAL_CKPT04}"
assert LUKI_RGB04.exists(),f"Missing Luki source RGB: {LUKI_RGB04}"

OUTPUT_TILE16=512      # output grid stride: no full-mosaic in-memory buffers
HALO16=64             # contextual overlap on each side (input size 640, multiple of 32)
BATCH16=4             # 4 x 640px fits typical L4 GPU; lower to 2 if needed
TEST_ROWS16=None      # e.g. 3 for a small test; set None for full Luki production
FORCE_NEW16=False     # True only to intentionally restart/replace incomplete output
OUT16=FINAL_RUN04/"luki_wall_to_wall"
OUT16.mkdir(parents=True,exist_ok=True)
PARTIAL16=OUT16/"Luki_M1_probability_uint16_NATIVE.partial.tif"
FINAL16=OUT16/"Luki_M1_probability_uint16_NATIVE.tif"
PROGRESS16=OUT16/"Luki_M1_inference_progress.json"
MANIFEST16=OUT16/"Luki_M1_inference_manifest.json"
SCALE16=10000
NODATA16=65535
VARIANT16="M1_native_photometric"

# Exact checkpoint, no guessing by filename or file modification time.
ckpt16=torch.load(FINAL_CKPT04,map_location="cpu",weights_only=False)
assert ckpt16.get("run_id")==FINAL_RUN_ID04 and ckpt16.get("variant")==VARIANT16, \
    "Checkpoint does not match final four-forest M1 run"
# Encoder architecture matches training; ImageNet weights need NOT be re-downloaded,
# since the complete trained checkpoint contains every encoder and decoder weight.
model16=smp.Unet(encoder_name="resnet18",encoder_weights=None,in_channels=3,classes=1)
model16.load_state_dict(ckpt16["model"],strict=True)
model16=model16.to("cuda").eval()
mean16=torch.tensor([.485,.456,.406],device="cuda").view(1,3,1,1)
std16=torch.tensor([.229,.224,.225],device="cuda").view(1,3,1,1)

with rasterio.open(LUKI_RGB04) as src:
    width16,height16=src.width,src.height
    assert src.count>=3 and src.dtypes[0]=="uint8", \
        "Validate unexpected Luki RGB channel type before inference."
    source_meta16=dict(width=src.width,height=src.height,
        crs=str(src.crs),transform=tuple(src.transform),
        source_size=LUKI_RGB04.stat().st_size,
        source_mtime_ns=LUKI_RGB04.stat().st_mtime_ns)
    profile16=src.profile.copy()

rows16=(height16+OUTPUT_TILE16-1)//OUTPUT_TILE16
cols16=(width16+OUTPUT_TILE16-1)//OUTPUT_TILE16
config16=dict(model_run=FINAL_RUN_ID04,checkpoint=str(FINAL_CKPT04),
    checkpoint_bytes=FINAL_CKPT04.stat().st_size,
    checkpoint_mtime_ns=FINAL_CKPT04.stat().st_mtime_ns,
    RGB_source=str(LUKI_RGB04),RGB_meta=source_meta16,
    output_tile=OUTPUT_TILE16,halo=HALO16,scale=SCALE16,
    nodata=NODATA16,normalization="ImageNet mean/std",
    input_bands=[1,2,3],
    excluded_RGB=["0/0/0","255/255/255","0/0/255"],
    no_alpha_or_raster_dataset_mask=True,illumination_gain="none",
    prediction="sigmoid binary Musanga model; context-halo tiled, no overlap-blending")
fingerprint16=hashlib.sha256(json.dumps(config16,sort_keys=True).encode()).hexdigest()[:16]
print("Native Luki image:",width16,"x",height16,"pixels")
print("Output windows:",rows16,"rows x",cols16,"columns =",rows16*cols16,"tiles")
print("Run:",FINAL_RUN_ID04,"| inference contract:",fingerprint16)
print("Output:",FINAL16)
print("Test mode:",TEST_ROWS16 is not None,"| GPU:",torch.cuda.get_device_name(0),flush=True)

if FINAL16.exists() and not FORCE_NEW16:
    assert MANIFEST16.exists(),"Found completed map without manifest; inspect before reusing"
    saved16=json.loads(MANIFEST16.read_text())
    assert saved16["fingerprint"]==fingerprint16, "Finished raster contract differs from requested run"
    print("REUSING FINISHED PROBABILITY MAP:",FINAL16,flush=True)
else:
    if FORCE_NEW16:
        for p in [PARTIAL16,PROGRESS16,MANIFEST16]:
            if p.exists():p.unlink()
        # Preserve completed prior outputs unless intentionally archived/renamed by user.
        assert not FINAL16.exists(),"Move/archive the completed map before requesting a new run."

    profile16.update(driver="GTiff",count=1,dtype="uint16",
        nodata=NODATA16,compress="deflate",predictor=2,
        tiled=True,blockxsize=512,blockysize=512,bigtiff="YES",
        interleave="band")
    if PARTIAL16.exists():
        assert PROGRESS16.exists(),"Partial raster exists without progress metadata; inspect before restarting."
        prev16=json.loads(PROGRESS16.read_text())
        assert prev16["fingerprint"]==fingerprint16,"Partial raster belongs to a different checkpoint/configuration"
        starting_row16=int(prev16["completed_output_rows"])
        print("RESUMING at completed output tile-row",starting_row16,flush=True)
        mode16="r+"
    else:
        starting_row16=0
        mode16="w"
    stopping_row16=rows16 if TEST_ROWS16 is None else min(rows16,starting_row16+int(TEST_ROWS16))

    def valid16(rgb):
        # Only source RGB; Luki alpha / dataset mask and nodata metadata are unreliable.
        return ~(np.all(rgb==0,axis=2)|np.all(rgb==255,axis=2)|
                 ((rgb[:,:,0]==0)&(rgb[:,:,1]==0)&(rgb[:,:,2]==255)))

    def tile16(src,x,y):
        # Input includes 64 px context, even outside mosaic (RGB padding = zero).
        # Output centre is exactly 512x512 at the native source transform.
        W=OUTPUT_TILE16+2*HALO16
        win=Window(x-HALO16,y-HALO16,W,W)
        rgb=np.moveaxis(src.read([1,2,3],window=win,boundless=True,fill_value=0),0,-1)
        central=rgb[HALO16:HALO16+OUTPUT_TILE16,HALO16:HALO16+OUTPUT_TILE16]
        return np.moveaxis(rgb,2,0),valid16(central)

    started16=time.monotonic()
    with rasterio.open(LUKI_RGB04) as src:
        with rasterio.open(PARTIAL16,mode16,**(profile16 if mode16=="w" else {})) as dst:
            for row16 in tqdm(range(starting_row16,stopping_row16),
                              desc="Luki M1 output tile-rows",unit="row"):
                y=row16*OUTPUT_TILE16
                for col0 in range(0,cols16,BATCH16):
                    batch=[];masks=[];positions=[]
                    for col in range(col0,min(cols16,col0+BATCH16)):
                        x=col*OUTPUT_TILE16
                        rgb,valid=tile16(src,x,y)
                        batch.append(rgb)
                        masks.append(valid)
                        positions.append((x,y,min(OUTPUT_TILE16,width16-x),
                                           min(OUTPUT_TILE16,height16-y)))
                    tensor=torch.from_numpy(np.stack(batch)).to("cuda",non_blocking=True).float()/255.
                    tensor=(tensor-mean16)/std16
                    with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=torch.float16):
                        pred=torch.sigmoid(model16(tensor)).float().cpu().numpy()[:,0]
                    for bi,(x,y,hwidth,hheight) in enumerate(positions):
                        center=pred[bi,HALO16:HALO16+hheight,
                                       HALO16:HALO16+hwidth]
                        mask=masks[bi][:hheight,:hwidth]
                        scaled=np.where(mask,np.rint(np.clip(center,0,1)*SCALE16),
                                        NODATA16).astype(np.uint16)
                        dst.write(scaled,1,window=Window(x,y,hwidth,hheight))
                    del tensor,pred,batch,masks
                # Only mark progress after the entire horizontal tile-row is saved.
                (PROGRESS16).write_text(json.dumps(dict(fingerprint=fingerprint16,
                    completed_output_rows=row16+1,expected_rows=rows16,
                    partial_output=str(PARTIAL16),last_updated_epoch=time.time()),indent=2))
                if (row16+1)%8==0 or row16==starting_row16:
                    elapsed=time.monotonic()-started16
                    print(f"OUTPUT PROGRESS: {row16+1}/{rows16} tile rows; "
                          f"elapsed {elapsed/60:.1f} min",flush=True)

    if stopping_row16<rows16:
        print(f"PARTIAL PREVIEW: {stopping_row16}/{rows16} tile rows saved; "
              "set TEST_ROWS16=None and rerun Section 16 to complete.",flush=True)
    else:
        # Check dimensions and source-grid equality after closing the writer.
        with rasterio.open(PARTIAL16) as written,rasterio.open(LUKI_RGB04) as src:
            assert (written.width,written.height)==(src.width,src.height)
            assert written.transform==src.transform and written.crs==src.crs
            assert written.nodata==NODATA16 and written.dtypes[0]=="uint16"
        os.replace(PARTIAL16,FINAL16)
        MANIFEST16.write_text(json.dumps(dict(
            fingerprint=fingerprint16,config=config16,
            output=str(FINAL16),all_tile_rows_completed=rows16,
            completed=True,source_native_grid_preserved=True,
            estimated_output_size_bytes=FINAL16.stat().st_size,
            elapsed_seconds=float(time.monotonic()-started16)),indent=2))
        if PROGRESS16.exists():PROGRESS16.unlink()
        print("COMPLETE LUKI FINAL M1 PROBABILITY MAP:",FINAL16,flush=True)
        print("DO NOT treat as illumination-corrected until spatial shadow QA is completed.",flush=True)


## 17 — Compare historical versus final M1 full-resolution Luki probability maps (CPU)

**Run directly after Section 16**, or after Section 12 with the completed M1 map on Drive. This reads the **historical frozen production Luki probability GeoTIFF** and the **new final M1 GeoTIFF** on the exact same pixel grid, using independent RGB-only validity masks. It performs a **complete streamed pixel census by default**, never loading the full 1.47-billion-pixel maps into RAM, and saves a compact CSV summary, probability histograms and four-panel maps (RGB, old, new, difference).

We compare *probabilities and thresholded predicted cover* on pixels valid in both maps, with no threshold retuning. Coverage may be affected by probability calibration; a change in the fraction of pixels with P ≥ 0.5 is **not automatically a real change in Musanga abundance**. We also summarize brightness versus probability in spatial windows to reveal systematic illumination relationships. Interpret the window correlations cautiously because tree-species composition also varies spatially.

For a quick smoke test choose `FULL_SCAN17=False` (approximately 150 spatially distributed windows); the default `True` measures all pixels. Neither image is altered; outputs go to `V2/04_illumination_robust_training/final_four_site_M1/<run_id>/luki_wall_to_wall/comparison/`.

In [ ]:
# Section 17: exact grid-aligned old vs M1 native Luki probability comparison.
# CPU-only, zero model inference and bounded-memory rasterio windows.
from pathlib import Path
import math,json
import numpy as np,pandas as pd,rasterio,matplotlib.pyplot as plt
from rasterio.windows import Window
from scipy.stats import spearmanr
from IPython.display import display
from tqdm.auto import tqdm

ROOT17=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
OLD17=ROOT17/"KVO_work/KVO_04_final_multisite/probability_maps/Luki_musanga_probability_uint16_FULL.tif"
NEW17=ROOT17/"KVO_work/V2/04_illumination_robust_training/final_four_site_M1/337073e7b74f/luki_wall_to_wall/Luki_M1_probability_uint16_NATIVE.tif"
RGB17=ROOT17/"data/Geo_Ref/Luki.tif"
OUT17=NEW17.parent/"comparison"
OUT17.mkdir(parents=True,exist_ok=True)
FULL_SCAN17=True      # False = fast, spatially distributed 150-window sample
SAMPLE_WINDOWS17=150
WINDOW17=512
PREVIEW17=1024
SCALE17=10000.
NODATA17=65535
THRESH17=.50
assert all(p.exists() for p in [OLD17,NEW17,RGB17]), \
    "Missing a raster; check OLD17, NEW17 and RGB17 paths."
def rgb_valid17(c):
    return ~(np.all(c==0,axis=0)|np.all(c==255,axis=0)|
             ((c[0]==0)&(c[1]==0)&(c[2]==255)))
def usable17(v,src):
    return (v!=NODATA17)&(v<=SCALE17)&np.isfinite(v)

# Stream window pairs to avoid >10GB in-memory arrays.
with rasterio.open(OLD17) as old,rasterio.open(NEW17) as new,rasterio.open(RGB17) as rgb:
    assert (old.width,old.height,old.transform,old.crs)== \
           (new.width,new.height,new.transform,new.crs), \
           "Original and new probability rasters are not on identical grids."
    assert (rgb.width,rgb.height,rgb.transform,rgb.crs)== \
           (new.width,new.height,new.transform,new.crs), \
           "RGB and probability grids differ; investigate before comparing."
    assert old.count==new.count==1
    assert old.dtypes[0]==new.dtypes[0]=="uint16"
    assert old.nodata==new.nodata==NODATA17, \
        "Unexpected no-data conventions: check rather than silently compare."
    nx=math.ceil(old.width/WINDOW17)
    ny=math.ceil(old.height/WINDOW17)
    all_windows=[(ix,iy) for iy in range(ny) for ix in range(nx)]
    if FULL_SCAN17:
        selected=all_windows
    else:
        indices=np.unique(np.linspace(0,len(all_windows)-1,
                                       min(SAMPLE_WINDOWS17,len(all_windows))).round().astype(int))
        selected=[all_windows[int(i)] for i in indices]
    print("Mode:", "FULL MAP CENSUS" if FULL_SCAN17 else "SPATIAL SUBSAMPLE")
    print("Output windows:",len(selected),"/",len(all_windows))
    totals=dict(compared_pixels=0,original_only_pixels=0,new_only_pixels=0,
                rgb_valid_pixels=0,old_sum=0.,new_sum=0.,difference_sum=0.,
                difference_squared_sum=0.,old_above=0,new_above=0,
                old_very_high=0,new_very_high=0)
    hist_old=np.zeros(50,dtype=np.int64)
    hist_new=np.zeros(50,dtype=np.int64)
    window_rows=[]
    for ix,iy in tqdm(selected,desc="Paired Luki maps (CPU windows)"):
        x0=ix*WINDOW17;y0=iy*WINDOW17
        w=min(WINDOW17,old.width-x0);h=min(WINDOW17,old.height-y0)
        win=Window(x0,y0,w,h)
        o=old.read(1,window=win)
        n=new.read(1,window=win)
        c=rgb.read([1,2,3],window=win)
        rv=rgb_valid17(c)
        ov=rv&usable17(o,old)
        nv=rv&usable17(n,new)
        valid=ov&nv
        totals["rgb_valid_pixels"]+=int(rv.sum())
        totals["original_only_pixels"]+=int((ov&~nv).sum())
        totals["new_only_pixels"]+=int((nv&~ov).sum())
        count=int(valid.sum())
        if count<100:continue
        old_p=o[valid].astype(np.float32)/SCALE17
        new_p=n[valid].astype(np.float32)/SCALE17
        diff=new_p-old_p
        totals["compared_pixels"]+=count
        totals["old_sum"]+=float(old_p.sum(dtype=np.float64))
        totals["new_sum"]+=float(new_p.sum(dtype=np.float64))
        totals["difference_sum"]+=float(diff.sum(dtype=np.float64))
        totals["difference_squared_sum"]+=float(np.square(diff).sum(dtype=np.float64))
        totals["old_above"]+=int((old_p>=THRESH17).sum())
        totals["new_above"]+=int((new_p>=THRESH17).sum())
        totals["old_very_high"]+=int((old_p>=.9).sum())
        totals["new_very_high"]+=int((new_p>=.9).sum())
        hist_old+=np.histogram(old_p,bins=50,range=(0,1))[0]
        hist_new+=np.histogram(new_p,bins=50,range=(0,1))[0]
        brightness=(.2126*c[0].astype(np.float32)+
                    .7152*c[1].astype(np.float32)+
                    .0722*c[2].astype(np.float32))/255.
        window_rows.append(dict(ix=ix,iy=iy,n=count,
             median_rgb_brightness=float(np.median(brightness[valid])),
             original_mean_p=float(old_p.mean()),M1_mean_p=float(new_p.mean()),
             mean_delta_p=float(diff.mean()),
             original_cover_0p5=float((old_p>=THRESH17).mean()),
             M1_cover_0p5=float((new_p>=THRESH17).mean())))
    assert totals["compared_pixels"]>0,"No jointly valid source pixels; verify map alignment and no-data."
    z=totals["compared_pixels"]
    summary17=dict(scope="all_RGB_and_both_model_valid_pixels" if FULL_SCAN17 else "sampled_RGB_and_both_model_valid_pixels",
        spatial_windows=len(selected),compared_pixels=z,
        valid_RGB_pixels=totals["rgb_valid_pixels"],
        original_only_valid_pixels=totals["original_only_pixels"],
        M1_only_valid_pixels=totals["new_only_pixels"],
        original_mean_probability=totals["old_sum"]/z,
        M1_mean_probability=totals["new_sum"]/z,
        mean_M1_minus_original=totals["difference_sum"]/z,
        RMSE_probability_difference=math.sqrt(totals["difference_squared_sum"]/z),
        original_pixel_fraction_ge_0p5=totals["old_above"]/z,
        M1_pixel_fraction_ge_0p5=totals["new_above"]/z,
        change_fraction_ge_0p5=(totals["new_above"]-totals["old_above"])/z,
        original_pixel_fraction_ge_0p9=totals["old_very_high"]/z,
        M1_pixel_fraction_ge_0p9=totals["new_very_high"]/z)
    df17=pd.DataFrame([summary17])
    windows17=pd.DataFrame(window_rows)
    df17.to_csv(OUT17/"17_Luki_map_comparison_summary.csv",index=False)
    windows17.to_csv(OUT17/"17_Luki_window_brightness_vs_probability.csv",index=False)
    print("Paired full-map probability and cover comparison:")
    display(df17.T.rename(columns={0:"value"}).round(5))
    if len(windows17)>5:
        print("Window-level Spearman: median RGB brightness vs mean Musanga probability")
        rho_rows=[]
        for name,col in [("Historical U-Net","original_mean_p"),("Final M1","M1_mean_p")]:
            rr=spearmanr(windows17.median_rgb_brightness,windows17[col])
            rho_rows.append(dict(model=name,n_windows=len(windows17),
                                 rho=float(rr.statistic),p_value=float(rr.pvalue)))
        display(pd.DataFrame(rho_rows).round(4))
    fig,ax=plt.subplots(figsize=(8,4))
    centers=(np.arange(50)+.5)/50
    ax.plot(centers,hist_old/z,label="Original")
    ax.plot(centers,hist_new/z,label="Final M1")
    ax.set(xlabel="Pixel P(Musanga)",ylabel="Fraction of compared valid pixels",
           title="Luki: matched-pixel probability distribution")
    ax.legend()
    fig.savefig(OUT17/"17_Luki_probability_distributions.png",dpi=150,bbox_inches="tight")
    plt.show()

    # Representative high-support window: choose from the middle of eligible sample windows.
    eligible=windows17.loc[windows17.n>=WINDOW17*WINDOW17*.65].copy()
    if eligible.empty:eligible=windows17.copy()
    if len(eligible):
        ordered=eligible.sort_values("median_rgb_brightness")
        picks=[ordered.iloc[int(q*(len(ordered)-1))] for q in [.15,.5,.85]]
        for j,rec in enumerate(picks,1):
            cx=int(rec.ix)*WINDOW17+WINDOW17//2
            cy=int(rec.iy)*WINDOW17+WINDOW17//2
            px=max(0,min(old.width-PREVIEW17,cx-PREVIEW17//2))
            py=max(0,min(old.height-PREVIEW17,cy-PREVIEW17//2))
            pw=Window(px,py,min(PREVIEW17,old.width-px),min(PREVIEW17,old.height-py))
            src_img=np.moveaxis(rgb.read([1,2,3],window=pw),0,-1)
            po=old.read(1,window=pw)
            pn=new.read(1,window=pw)
            valid=rgb_valid17(np.moveaxis(src_img,2,0))&usable17(po,old)&usable17(pn,new)
            a=np.where(valid,po.astype(np.float32)/SCALE17,np.nan)
            b=np.where(valid,pn.astype(np.float32)/SCALE17,np.nan)
            fig,axes=plt.subplots(1,4,figsize=(17,4),constrained_layout=True)
            axes[0].imshow(src_img);axes[0].set_title("Luki RGB")
            for ax,im,title in [(axes[1],a,"Historical probability"),
                                (axes[2],b,"Final M1 probability")]:
                ax.imshow(im,cmap="viridis",vmin=0,vmax=1);ax.set_title(title)
            axes[3].imshow(b-a,cmap="coolwarm",vmin=-1,vmax=1)
            axes[3].set_title("M1 − historical")
            for ax in axes:ax.axis("off")
            fig.savefig(OUT17/f"17_Luki_matched_RGB_old_M1_difference_{j}.png",
                        dpi=140,bbox_inches="tight")
            plt.show()
    (OUT17/"17_comparison_manifest.json").write_text(json.dumps(dict(
        original_probability=str(OLD17),M1_probability=str(NEW17),
        source_RGB=str(RGB17),full_pixel_census=FULL_SCAN17,
        compared_pixels=z,threshold=THRESH17,summary=summary17),indent=2))
    print("SAVED CPU-ONLY LUKI MAP COMPARISON:",OUT17,flush=True)
    print("Fractions above 0.5 are thresholded probability-map fractions, NOT validated species cover.")
